In [2]:
SEED = 1
MODEL_NAMES = ("r18", "r50", "m3", "oracle")

In [1]:
from google.colab import drive
from pathlib import Path
import json
import torch

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
assert PROJECT.is_dir(), f"Missing project: {PROJECT}"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

config_fields = (
    "run_name", "seed", "model", "epochs", "batch_size",
    "learning_rate", "weight_decay", "optimizer",
    "initial_weights", "augmentation", "precision",
    "num_workers", "selection_metric", "selection_tie_rule",
    "epoch_seed_rule", "geometry_source", "failure_policy",
    "cudnn_deterministic", "cudnn_benchmark",
)

for run_name in (
    "r18_seed0", "r50_seed0", "m3_seed0", "oracle_seed0",
):
    run_root = PROJECT / "runs" / run_name
    print(f"\n===== {run_name} =====")
    print("Run directory exists:", run_root.is_dir())

    if not run_root.is_dir():
        continue

    print("Top-level files:")
    for path in sorted(run_root.iterdir()):
        print(" ", path.name, "[directory]" if path.is_dir() else "[file]")

    config_path = run_root / "config.json"
    if config_path.is_file():
        config = json.loads(config_path.read_text(encoding="utf-8"))
        print("All configuration keys:", sorted(config))
        print("Training configuration:")
        print(json.dumps(
            {key: config[key] for key in config_fields if key in config},
            indent=2,
        ))
    else:
        print("Top-level config.json: NOT FOUND")

    source_root = run_root / "source"
    print("Source snapshot files:")
    if source_root.is_dir():
        for path in sorted(source_root.rglob("*")):
            if path.is_file() and path.suffix in (
                ".py", ".json", ".yaml", ".yml",
            ):
                print(" ", path.relative_to(source_root))
    else:
        print("  No source snapshot directory")

print("\n===== Project notebooks and training scripts =====")
for directory in ("notebooks", "scripts", "training"):
    folder = PROJECT / directory
    print(f"{directory}: exists={folder.is_dir()}")
    if folder.is_dir():
        for path in sorted(folder.iterdir()):
            if path.is_file() and path.suffix in (".py", ".ipynb"):
                print(" ", path.name)

print("\n===== Existing Seed1 directories =====")
for run_name in (
    "r18_seed1", "r50_seed1", "m3_seed1", "oracle_seed1",
):
    path = PROJECT / "runs" / run_name
    print(run_name, "exists:", path.exists())

print("\nSeed1 reuse inspection: COMPLETE")
print("No training or file modifications performed")

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4

===== r18_seed0 =====
Run directory exists: True
Top-level files:
  best.json [file]
  config.json [file]
  day7 [directory]
  epoch_01.pt [file]
  epoch_02.pt [file]
  epoch_03.pt [file]
  epoch_04.pt [file]
  epoch_05.pt [file]
  epoch_06.pt [file]
  epoch_07.pt [file]
  epoch_08.pt [file]
  epoch_09.pt [file]
  epoch_10.pt [file]
  epoch_metrics.csv [file]
  history.json [file]
  source [directory]
  training_curves.png [file]
All configuration keys: ['attribute_names', 'augmentation', 'batch_size', 'cudnn_benchmark', 'cudnn_deterministic', 'environment', 'epoch_seed_rule', 'epochs', 'f1_threshold', 'git_commit', 'initial_weights', 'learning_rate', 'num_workers', 'optimizer', 'precision', 'seed', 'selection_metric', 'source_sha256', 'weight_decay']
Training configuration:
{
  "seed": 0,
  "epochs": 10,
  "batch_size": 64,
  "learning_rate": 0.0001,
  "weight_decay": 0.0,
  "optimizer": "Adam",
  "initial_weights": "ResNet

In [2]:
from google.colab import files
from pathlib import Path
import hashlib
import json
import zipfile

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
archive_path = Path("/content/seed0_training_reference.zip")
payloads = {}

for run_name in (
    "r18_seed0", "r50_seed0", "m3_seed0", "oracle_seed0",
):
    run_root = PROJECT / "runs" / run_name

    for filename in ("config.json", "history.json", "best.json"):
        path = run_root / filename
        assert path.is_file(), f"Missing training record: {path}"
        payloads[f"{run_name}/{filename}"] = path.read_bytes()

    source_root = run_root / "source"
    assert source_root.is_dir(), f"Missing source snapshot: {source_root}"

    for path in sorted(source_root.rglob("*")):
        if path.is_file() and path.suffix in (
            ".py", ".json", ".yaml", ".yml",
        ):
            relative = path.relative_to(source_root).as_posix()
            payloads[f"{run_name}/source/{relative}"] = path.read_bytes()

manifest = {
    "purpose": "Read-only historical training reference for Seed1 preparation",
    "files": {
        name: hashlib.sha256(data).hexdigest()
        for name, data in payloads.items()
    },
    "checkpoints_included": False,
    "images_included": False,
    "predictions_included": False,
}

payloads["reference_manifest.json"] = (
    json.dumps(manifest, indent=2) + "\n"
).encode("utf-8")

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name, data in sorted(payloads.items()):
        archive.writestr(name, data)

print("Reference files:", len(payloads))
print("Archive:", archive_path)
print("Historical runs unchanged")
print("No training performed")
files.download(str(archive_path))

Reference files: 35
Archive: /content/seed0_training_reference.zip
Historical runs unchanged
No training performed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import importlib.metadata
import platform
from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")

print("Python:", platform.python_version())
for package in (
    "torch", "torchvision", "numpy", "Pillow", "scikit-learn",
):
    try:
        print(package + ":", importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print(package + ": NOT INSTALLED")

print("CUDA runtime:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for relative in (
    "data/img_align_celeba.zip",
    "data/list_attr_celeba.txt",
    "data/list_eval_partition.txt",
    "data/list_landmarks_align_celeba.txt",
    "cache/mtcnn_clean.sqlite",
):
    path = PROJECT / relative
    print(relative, "| exists:", path.is_file())

print("Seed1 environment inspection: COMPLETE")
print("No training or dependency changes performed")

Python: 3.13.15
torch: 2.11.0+cu130
torchvision: 0.26.0+cu130
numpy: 2.1.3
Pillow: 11.3.0
scikit-learn: 1.6.1
CUDA runtime: 13.0
cuDNN: 92700
CUDA available: True
GPU: Tesla T4
data/img_align_celeba.zip | exists: True
data/list_attr_celeba.txt | exists: True
data/list_eval_partition.txt | exists: True
data/list_landmarks_align_celeba.txt | exists: True
cache/mtcnn_clean.sqlite | exists: True
Seed1 environment inspection: COMPLETE
No training or dependency changes performed


In [4]:
from pathlib import Path
import hashlib
import json
import shutil
import zipfile

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
DATA_ROOT = Path("/content/seed1_shared_data/raw")
WORKSPACE_ROOT = Path("/content/seed1_workspaces")

assert PROJECT.is_dir(), f"Project not found: {PROJECT}"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def copy_verified(source, destination):
    source = Path(source)
    destination = Path(destination)
    assert source.is_file(), f"Missing source: {source}"
    source_hash = sha_file(source)

    if destination.exists():
        assert sha_file(destination) == source_hash, (
            f"Existing file differs; preserved: {destination}"
        )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + ".partial")
        shutil.copyfile(source, temporary)
        assert sha_file(temporary) == source_hash, (
            f"Copy verification failed: {source}"
        )
        temporary.replace(destination)

    return source_hash


# Copy annotation files once for all four runs.
annotation_hashes = {}
for filename in (
    "list_attr_celeba.txt",
    "list_eval_partition.txt",
    "list_landmarks_align_celeba.txt",
):
    annotation_hashes[filename] = copy_verified(
        PROJECT / "data" / filename,
        DATA_ROOT / filename,
    )

partitions = {"0": [], "1": [], "2": []}
with (DATA_ROOT / "list_eval_partition.txt").open(
    encoding="utf-8-sig"
) as stream:
    for line in stream:
        filename, split_code = line.split()
        partitions[split_code].append(filename)

assert {key: len(value) for key, value in partitions.items()} == {
    "0": 162770,
    "1": 19867,
    "2": 19962,
}

all_names = [name for names in partitions.values() for name in names]
assert len(all_names) == len(set(all_names)), "Duplicate partition IDs"

required_names = partitions["0"] + partitions["1"]
assert len(required_names) == 182637

# Reuse the original archive without downloading it again for each model.
local_zip = DATA_ROOT.parent / "img_align_celeba.zip"
print("Preparing shared image archive...")
image_zip_hash = copy_verified(
    PROJECT / "data/img_align_celeba.zip",
    local_zip,
)
assert image_zip_hash == (
    "46fb89443c578308acf364d7d379fe1b9efb"
    "793042c0af734b6112e4fd3a8c74"
), "Image archive differs from the historical benchmark archive"

image_directory = DATA_ROOT / "img_align_celeba"
image_directory.mkdir(parents=True, exist_ok=True)

# Reading each ZIP member checks its CRC. Existing images must match exactly.
with zipfile.ZipFile(local_zip) as archive:
    members = {}
    for member in archive.infolist():
        if member.is_dir():
            continue
        filename = Path(member.filename).name
        if filename.endswith(".jpg"):
            assert filename not in members, f"Duplicate ZIP image: {filename}"
            members[filename] = member

    assert set(required_names).issubset(members), "Required images are missing"

    for index, filename in enumerate(required_names, start=1):
        data = archive.read(members[filename])
        destination = image_directory / filename

        if destination.exists():
            assert destination.read_bytes() == data, (
                f"Existing image differs; preserved: {filename}"
            )
        else:
            temporary = destination.with_name(filename + ".partial")
            temporary.write_bytes(data)
            temporary.replace(destination)

        if index % 20000 == 0 or index == len(required_names):
            print(f"Verified train/validation images: {index}/{len(required_names)}")

assert all((image_directory / name).is_file() for name in required_names)
print("Official train/validation membership and image bytes: PASS")

# Keep each model's historical implementation in a separate workspace.
SEED1_WORKSPACES = {}
reference_config_hashes = {}

for model_name in ("r18", "r50", "m3", "oracle"):
    reference_name = f"{model_name}_seed0"
    run_name = f"{model_name}_seed1"
    reference_root = PROJECT / "runs" / reference_name
    config_path = reference_root / "config.json"
    config = json.loads(config_path.read_text(encoding="utf-8"))

    assert config["seed"] == 0
    assert config["epochs"] == 10
    assert config["batch_size"] == 64
    assert config["precision"] == "FP32"

    workspace = WORKSPACE_ROOT / run_name
    workspace.mkdir(parents=True, exist_ok=True)

    for relative, expected_hash in config["source_sha256"].items():
        relative_path = Path(relative)
        assert not relative_path.is_absolute() and ".." not in relative_path.parts

        source = reference_root / "source" / relative_path
        assert sha_file(source) == expected_hash, (
            f"Historical source differs: {reference_name}/{relative}"
        )
        actual_hash = copy_verified(source, workspace / relative_path)
        assert actual_hash == expected_hash

    SEED1_WORKSPACES[run_name] = workspace
    reference_config_hashes[reference_name] = sha_file(config_path)
    print(f"{run_name}: historical source snapshot verified")

cache_path = PROJECT / "cache/mtcnn_clean.sqlite"
assert cache_path.is_file(), f"Missing landmark cache: {cache_path}"

preparation = {
    "image_zip_sha256": image_zip_hash,
    "annotation_sha256": annotation_hashes,
    "reference_config_sha256": reference_config_hashes,
    "train_samples": len(partitions["0"]),
    "validation_samples": len(partitions["1"]),
    "training_started": False,
    "test_evaluated": False,
}
record_path = DATA_ROOT.parent / "preparation.json"
record_bytes = (
    json.dumps(preparation, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

if record_path.exists():
    assert record_path.read_bytes() == record_bytes, (
        "Existing preparation record differs; preserved"
    )
else:
    record_path.write_bytes(record_bytes)

print("Shared data root:", DATA_ROOT)
print("Source workspaces:", WORKSPACE_ROOT)
print("M3 clean landmark cache:", cache_path)
print("Seed1 shared data and source preparation: PASS")
print("No optimizer updates or test evaluation performed")

Preparing shared image archive...
Verified train/validation images: 20000/182637
Verified train/validation images: 40000/182637
Verified train/validation images: 60000/182637
Verified train/validation images: 80000/182637
Verified train/validation images: 100000/182637
Verified train/validation images: 120000/182637
Verified train/validation images: 140000/182637
Verified train/validation images: 160000/182637
Verified train/validation images: 180000/182637
Verified train/validation images: 182637/182637
Official train/validation membership and image bytes: PASS
r18_seed1: historical source snapshot verified
r50_seed1: historical source snapshot verified
m3_seed1: historical source snapshot verified
oracle_seed1: historical source snapshot verified
Shared data root: /content/seed1_shared_data/raw
Source workspaces: /content/seed1_workspaces
M3 clean landmark cache: /content/drive/MyDrive/FacialVisualProfile/cache/mtcnn_clean.sqlite
Seed1 shared data and source preparation: PASS
No opti

In [5]:
from pathlib import Path
import copy
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import random
import sys

import numpy as np
import torch
from torch import nn

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
DATA_ROOT = Path("/content/seed1_shared_data/raw")
WORKSPACE = Path("/content/seed1_workspaces/r18_seed1")
RUN_ROOT = PROJECT / "runs/r18_seed1"
REFERENCE_ROOT = PROJECT / "runs/r18_seed0"

assert torch.cuda.is_available(), "A GPU runtime is required"
assert WORKSPACE.is_dir(), f"Missing workspace: {WORKSPACE}"
assert (DATA_ROOT.parent / "preparation.json").is_file()


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(
        json.dumps(payload, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


reference_config_path = REFERENCE_ROOT / "config.json"
reference_config = json.loads(
    reference_config_path.read_text(encoding="utf-8")
)

# Verify and archive the exact historical source files.
RUN_ROOT.mkdir(parents=True, exist_ok=True)
for relative, expected_hash in reference_config["source_sha256"].items():
    source = WORKSPACE / relative
    assert sha_file(source) == expected_hash, (
        f"Historical source differs: {relative}"
    )
    destination = RUN_ROOT / "source" / relative
    data = source.read_bytes()

    if destination.exists():
        assert destination.read_bytes() == data, (
            f"Existing source differs; preserved: {destination}"
        )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(data)

# Resolve imports only from the verified R18 workspace.
for package in ("datasets", "models", "evaluation"):
    package_directory = WORKSPACE / package
    assert package_directory.is_dir()
    initializer = package_directory / "__init__.py"
    if not initializer.exists():
        initializer.write_text("", encoding="utf-8")

    for module_name in list(sys.modules):
        if module_name == package or module_name.startswith(package + "."):
            del sys.modules[module_name]

os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
importlib.invalidate_caches()

from datasets.celeba import CelebAAttributes
from models.resnet18 import ResNet18Attributes
from evaluation.attributes import evaluate_attributes

for module_name in (
    "datasets.celeba",
    "models.resnet18",
    "evaluation.attributes",
):
    module_path = Path(sys.modules[module_name].__file__).resolve()
    assert module_path.is_relative_to(WORKSPACE.resolve()), (
        f"Unexpected module source: {module_path}"
    )

environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": importlib.metadata.version("torchvision"),
    "numpy": np.__version__,
    "Pillow": importlib.metadata.version("Pillow"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}

CONFIG = copy.deepcopy(reference_config)
CONFIG.pop("git_commit", None)
CONFIG.update({
    "run_name": "r18_seed1",
    "seed": 1,
    "environment": environment,
    "reference_run": "r18_seed0",
    "reference_config_sha256": sha_file(reference_config_path),
    "reference_source_commit": reference_config["git_commit"],
    "selection_tie_rule": "earliest epoch",
    "data_preparation_sha256": sha_file(
        DATA_ROOT.parent / "preparation.json"
    ),
    "runtime": {
        "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "gpu": torch.cuda.get_device_name(0),
    },
    "reference_environment": reference_config["environment"],
})

config_path = RUN_ROOT / "config.json"
if config_path.exists():
    saved_config = json.loads(config_path.read_text(encoding="utf-8"))
    assert saved_config == CONFIG, (
        "Existing run configuration differs; preserved"
    )
else:
    write_json_atomic(config_path, CONFIG)

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
torch.cuda.manual_seed_all(CONFIG["seed"])
torch.backends.cudnn.deterministic = CONFIG["cudnn_deterministic"]
torch.backends.cudnn.benchmark = CONFIG["cudnn_benchmark"]

TRAIN_DATASET = CelebAAttributes(split="train", root=DATA_ROOT)
VALID_DATASET = CelebAAttributes(split="valid", root=DATA_ROOT)

assert len(TRAIN_DATASET) == 162770
assert len(VALID_DATASET) == 19867
assert TRAIN_DATASET.attribute_names == CONFIG["attribute_names"]
assert VALID_DATASET.attribute_names == CONFIG["attribute_names"]

DEVICE = torch.device("cuda")
checkpoints = sorted(RUN_ROOT.glob("epoch_[0-9][0-9].pt"))

# Reuse only this Seed1 run when resuming.
MODEL = ResNet18Attributes(
    num_attributes=len(CONFIG["attribute_names"]),
    pretrained=not bool(checkpoints),
).to(DEVICE)

OPTIMIZER = torch.optim.Adam(
    MODEL.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
CRITERION = nn.BCEWithLogitsLoss()

if checkpoints:
    checkpoint_path = checkpoints[-1]
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )
    assert checkpoint["config"] == CONFIG
    MODEL.load_state_dict(checkpoint["model_state"], strict=True)
    OPTIMIZER.load_state_dict(checkpoint["optimizer_state"])
    HISTORY = checkpoint["history"]
    START_EPOCH = checkpoint["epoch"] + 1
    print("Restored Seed1 checkpoint:", checkpoint_path.name)
else:
    HISTORY = []
    START_EPOCH = 1
    checkpoint_path = RUN_ROOT / "epoch_00.pt"
    temporary = checkpoint_path.with_name(checkpoint_path.name + ".partial")
    torch.save({
        "epoch": 0,
        "model_state": MODEL.state_dict(),
        "optimizer_state": OPTIMIZER.state_dict(),
        "config": CONFIG,
        "history": HISTORY,
    }, temporary)
    temporary.replace(checkpoint_path)
    print("Fresh ImageNet initialization saved:", checkpoint_path.name)

# Evaluate a small batch without updating parameters or BatchNorm statistics.
MODEL.eval()
images = torch.stack([VALID_DATASET[i][0] for i in range(4)])
with torch.inference_mode():
    logits = MODEL(images.to(DEVICE))
assert logits.shape == (4, 24)
assert torch.isfinite(logits).all()
assert all(parameter.requires_grad for parameter in MODEL.parameters())

print("Dataset sizes:", len(TRAIN_DATASET), len(VALID_DATASET))
print("Forward shape:", tuple(logits.shape))
print("Next training epoch:", START_EPOCH)
print("Run directory:", RUN_ROOT)
print("Reference environment:", reference_config["environment"])
print("Current environment:", environment)
print("R18 Seed1 initialization and forward check: PASS")
print("No optimizer updates or test evaluation performed")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 117MB/s]


Fresh ImageNet initialization saved: epoch_00.pt
Dataset sizes: 162770 19867
Forward shape: (4, 24)
Next training epoch: 1
Run directory: /content/drive/MyDrive/FacialVisualProfile/runs/r18_seed1
Reference environment: {'python': '3.13.15', 'torch': '2.11.0+cu128', 'torchvision': '0.26.0+cu128', 'numpy': '2.1.3', 'Pillow': '11.3.0', 'scikit-learn': '1.6.1'}
Current environment: {'python': '3.13.15', 'torch': '2.11.0+cu130', 'torchvision': '0.26.0+cu130', 'numpy': '2.1.3', 'Pillow': '11.3.0', 'scikit-learn': '1.6.1'}
R18 Seed1 initialization and forward check: PASS
No optimizer updates or test evaluation performed


In [8]:
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def train_r18(
    model,
    optimizer,
    criterion,
    train_dataset,
    valid_dataset,
    config,
    project,
    device,
    metric_function,
):
    run_name = f"r18_seed{config['seed']}"
    assert config["run_name"] == run_name
    run_root = Path(project) / "runs" / run_name
    assert run_root.is_dir()

    saved_config = json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    )
    assert saved_config == config, "Run configuration mismatch"

    for relative, expected_hash in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected_hash, (
            f"Archived source differs: {relative}"
        )

    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def make_loader(dataset, shuffle, epoch_seed):
        return DataLoader(
            dataset,
            batch_size=config["batch_size"],
            shuffle=shuffle,
            generator=torch.Generator().manual_seed(epoch_seed),
            num_workers=config["num_workers"],
            worker_init_fn=seed_worker,
            pin_memory=True,
            drop_last=False,
        )

    def save_summaries(history):
        write_json_atomic(run_root / "history.json", history)
        if not history:
            return None

        # Python max retains the earliest record when scores are equal.
        best = max(history, key=lambda row: row["validation"]["map"])
        summary = {
            "epoch": best["epoch"],
            "checkpoint": f"epoch_{best['epoch']:02d}.pt",
            "validation_map": best["validation"]["map"],
            "validation_macro_f1": best["validation"]["macro_f1"],
        }
        write_json_atomic(run_root / "best.json", summary)
        return summary

    # Reload saved state on every invocation, including interrupted reruns.
    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Initialization checkpoint is missing"
    latest = checkpoints[-1]

    from torch.torch_version import TorchVersion

    # Allow the version object stored in this run's initialization checkpoint.
    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )

    # Store plain strings in subsequent checkpoints.
    config["environment"]["torch"] = str(config["environment"]["torch"])
    checkpoint["config"]["environment"]["torch"] = str(
        checkpoint["config"]["environment"]["torch"]
    )

    assert checkpoint["config"] == config, "Checkpoint configuration mismatch"

    model.load_state_dict(checkpoint["model_state"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    history = checkpoint["history"]
    completed_epoch = checkpoint["epoch"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    ), "Checkpoint history is not continuous"
    assert 0 <= completed_epoch <= config["epochs"]
    del checkpoint

    save_summaries(history)
    print("Run:", run_name, flush=True)
    print("Restored:", latest.name, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        train_loader = make_loader(train_dataset, True, epoch_seed)
        valid_loader = make_loader(valid_dataset, False, epoch_seed)

        torch.cuda.synchronize()
        started = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_count = 0

        progress = tqdm(
            train_loader,
            desc=f"{run_name} epoch {epoch:02d} train",
        )
        for images, targets, _, _ in progress:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            assert torch.isfinite(loss), "Non-finite training loss"

            loss.backward()
            optimizer.step()

            count = images.size(0)
            train_loss_sum += loss.item() * count
            train_count += count

            if train_count % (config["batch_size"] * 50) == 0:
                progress.set_postfix(
                    BCE=f"{train_loss_sum / train_count:.6f}"
                )

        assert train_count == len(train_dataset)

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        logits_parts = []
        target_parts = []

        with torch.inference_mode():
            for images, targets, _, _ in tqdm(
                valid_loader,
                desc=f"{run_name} epoch {epoch:02d} valid",
            ):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                logits = model(images)
                loss = criterion(logits, targets)

                assert torch.isfinite(logits).all(), (
                    "Non-finite validation logits"
                )
                assert torch.isfinite(loss), "Non-finite validation loss"

                count = images.size(0)
                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu())
                target_parts.append(targets.cpu())

        assert valid_count == len(valid_dataset)
        metrics = metric_function(
            torch.cat(logits_parts).numpy(),
            torch.cat(target_parts).numpy(),
            threshold=config["f1_threshold"],
        )
        del logits_parts, target_parts

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_loss": train_loss_sum / train_count,
            "valid_loss": valid_loss_sum / valid_count,
            "validation": metrics,
            "seconds_excluding_save": time.perf_counter() - started,
        }
        history.append(record)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": config,
            "history": history,
        }, temporary)
        temporary.replace(checkpoint_path)
        best = save_summaries(history)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_loss']:.6f} | "
            f"valid BCE={record['valid_loss']:.6f} | "
            f"F1@0.5={metrics['macro_f1']:.6f} | "
            f"mAP={metrics['map']:.6f} | "
            f"{record['seconds_excluding_save'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )

    best = save_summaries(history)
    print("Selected checkpoint:", run_root / best["checkpoint"])
    print("R18 training: COMPLETE")
    print("Clean-validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best


HISTORY, BEST = train_r18(
    model=MODEL,
    optimizer=OPTIMIZER,
    criterion=CRITERION,
    train_dataset=TRAIN_DATASET,
    valid_dataset=VALID_DATASET,
    config=CONFIG,
    project=PROJECT,
    device=DEVICE,
    metric_function=evaluate_attributes,
)

Run: r18_seed1
Restored: epoch_00.pt
Next epoch: 1


r18_seed1 epoch 01 train:   0%|          | 0/2544 [00:00<?, ?it/s]

r18_seed1 epoch 01 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 01 saved | train BCE=0.212885 | valid BCE=0.196006 | F1@0.5=0.731980 | mAP=0.809882 | 11.6 min | best epoch=1


r18_seed1 epoch 02 train:   0%|          | 0/2544 [00:00<?, ?it/s]

...

r18_seed1 epoch 09 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 09 saved | train BCE=0.053456 | valid BCE=0.331573 | F1@0.5=0.720906 | mAP=0.776254 | 12.2 min | best epoch=3


r18_seed1 epoch 10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

r18_seed1 epoch 10 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 10 saved | train BCE=0.041957 | valid BCE=0.355548 | F1@0.5=0.719135 | mAP=0.773384 | 12.8 min | best epoch=3
Selected checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/r18_seed1/epoch_03.pt
R18 training: COMPLETE
Clean-validation prediction export and threshold fitting remain pending
No test evaluation performed


In [9]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def export_r18_validation(
    model, dataset, config, project, device, metric_function
):
    run_root = Path(project) / "runs" / config["run_name"]
    assert config["run_name"] == f"r18_seed{config['seed']}"

    saved_config = json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    )
    assert saved_config == config, "Run configuration mismatch"

    best = json.loads(
        (run_root / "best.json").read_text(encoding="utf-8")
    )
    checkpoint_path = run_root / best["checkpoint"]

    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )
    assert checkpoint["config"] == config
    assert checkpoint["epoch"] == best["epoch"]

    selected = max(
        checkpoint["history"],
        key=lambda row: row["validation"]["map"],
    )
    assert selected["epoch"] == best["epoch"]
    model.load_state_dict(checkpoint["model_state"], strict=True)
    del checkpoint

    model.eval()
    loader = DataLoader(
        dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        generator=torch.Generator().manual_seed(
            config["seed"] + best["epoch"]
        ),
        num_workers=config["num_workers"],
        worker_init_fn=seed_worker,
        pin_memory=True,
        drop_last=False,
    )

    logits_parts = []
    target_parts = []
    filenames = []

    with torch.inference_mode():
        for images, targets, _, names in tqdm(
            loader, desc=f"{config['run_name']} selected validation"
        ):
            logits = model(images.to(device, non_blocking=True))
            assert torch.isfinite(logits).all()
            logits_parts.append(logits.cpu().numpy())
            target_parts.append(targets.numpy())
            filenames.extend(names)

    arrays = {
        "filenames": np.asarray(filenames),
        "logits": np.concatenate(logits_parts),
        "targets": np.concatenate(target_parts),
        "attribute_names": np.asarray(dataset.attribute_names),
    }

    assert filenames == dataset.names, "Validation order differs"
    assert len(filenames) == len(set(filenames)) == 19867
    assert arrays["logits"].shape == (19867, 24)
    assert arrays["targets"].shape == (19867, 24)
    assert dataset.attribute_names == config["attribute_names"]

    expected_targets = torch.stack([
        dataset.targets[name] for name in filenames
    ]).numpy()
    assert np.array_equal(arrays["targets"], expected_targets)

    metrics = metric_function(
        arrays["logits"], arrays["targets"], threshold=0.5
    )
    assert abs(
        metrics["macro_f1"] - best["validation_macro_f1"]
    ) <= 1e-10, "Selected-checkpoint F1 reproduction failed"
    assert abs(
        metrics["map"] - best["validation_map"]
    ) <= 1e-10, "Selected-checkpoint mAP reproduction failed"

    prediction_path = run_root / "validation_predictions.npz"
    if prediction_path.exists():
        with np.load(prediction_path, allow_pickle=False) as existing:
            assert set(existing.files) == set(arrays)
            for key, value in arrays.items():
                assert np.array_equal(existing[key], value), (
                    f"Existing predictions differ; preserved: {key}"
                )
    else:
        temporary = prediction_path.with_name(
            prediction_path.name + ".partial"
        )
        with temporary.open("wb") as stream:
            np.savez_compressed(stream, **arrays)
        temporary.replace(prediction_path)

    report = {
        "run_name": config["run_name"],
        "selected_epoch": best["epoch"],
        "checkpoint": best["checkpoint"],
        "checkpoint_sha256": sha_file(checkpoint_path),
        "config_sha256": sha_file(run_root / "config.json"),
        "prediction_sha256": sha_file(prediction_path),
        "split": "valid",
        "samples": len(filenames),
        "attribute_names": config["attribute_names"],
        "metrics": metrics,
        "threshold_fitted": False,
        "test_evaluated": False,
        "passed": True,
    }

    report_path = run_root / "validation_export_audit.json"
    if report_path.exists():
        assert json.loads(
            report_path.read_text(encoding="utf-8")
        ) == report, "Existing export audit differs; preserved"
    else:
        write_json_atomic(report_path, report)

    print("Full validation IDs, targets and attribute order: PASS")
    print("Selected-checkpoint metric reproduction: PASS")
    print(f"F1@0.5: {metrics['macro_f1']:.6f}")
    print(f"mAP: {metrics['map']:.6f}")
    print("Saved:", prediction_path)
    print("No threshold fitting or test evaluation performed")
    return report


VALIDATION_EXPORT = export_r18_validation(
    model=MODEL,
    dataset=VALID_DATASET,
    config=CONFIG,
    project=PROJECT,
    device=DEVICE,
    metric_function=evaluate_attributes,
)

r18_seed1 selected validation:   0%|          | 0/311 [00:00<?, ?it/s]

Full validation IDs, targets and attribute order: PASS
Selected-checkpoint metric reproduction: PASS
F1@0.5: 0.739332
mAP: 0.819523
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/r18_seed1/validation_predictions.npz
No threshold fitting or test evaluation performed


In [10]:
import json
from pathlib import Path

import numpy as np
from sklearn.metrics import f1_score


def freeze_clean_thresholds(config, project, metric_function):
    run_name = config["run_name"]
    run_root = Path(project) / "runs" / run_name
    prediction_path = run_root / "validation_predictions.npz"
    audit_path = run_root / "validation_export_audit.json"

    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    assert audit["passed"] is True
    assert audit["split"] == "valid"
    assert audit["test_evaluated"] is False
    assert audit["run_name"] == run_name
    assert sha_file(prediction_path) == audit["prediction_sha256"]
    assert sha_file(run_root / "config.json") == audit["config_sha256"]
    assert json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    ) == config

    checkpoint_path = run_root / audit["checkpoint"]
    assert sha_file(checkpoint_path) == audit["checkpoint_sha256"]

    with np.load(prediction_path, allow_pickle=False) as saved:
        filenames = saved["filenames"]
        logits = saved["logits"]
        targets = saved["targets"]
        attribute_names = saved["attribute_names"].tolist()

    assert filenames.shape == (19867,)
    assert len(set(filenames.tolist())) == 19867
    assert logits.shape == targets.shape == (19867, 24)
    assert np.isfinite(logits).all()
    assert np.isin(targets, [0, 1]).all()
    assert attribute_names == config["attribute_names"]
    assert all(np.unique(targets[:, i]).size == 2 for i in range(24))

    # Check IDs, labels and attribute order against historical validation.
    reference_path = (
        Path(project) / "runs/r18_seed0/day7/validation_predictions.npz"
    )
    with np.load(reference_path, allow_pickle=False) as reference:
        assert np.array_equal(filenames, reference["filenames"])
        assert np.array_equal(targets, reference["targets"])
        assert attribute_names == reference["attribute_names"].tolist()

    baseline = metric_function(logits, targets, threshold=0.5)
    for key in ("macro_f1", "map"):
        assert abs(baseline[key] - audit["metrics"][key]) <= 1e-10

    # Use the same stable float64 sigmoid as the historical evaluation.
    values = logits.astype(np.float64)
    probabilities = np.empty_like(values)
    positive = values >= 0
    probabilities[positive] = 1 / (1 + np.exp(-values[positive]))
    exponentials = np.exp(values[~positive])
    probabilities[~positive] = exponentials / (1 + exponentials)

    grid_integers = list(range(5, 96))
    grid = [value / 100 for value in grid_integers]
    thresholds = []
    tuned_f1 = []

    for attribute_index in range(24):
        scores = [
            float(f1_score(
                targets[:, attribute_index],
                probabilities[:, attribute_index] >= threshold,
                zero_division=0,
            ))
            for threshold in grid
        ]
        selected_index = min(
            range(len(grid)),
            key=lambda index: (
                -scores[index],
                abs(grid_integers[index] - 50),
                grid_integers[index],
            ),
        )
        thresholds.append(grid[selected_index])
        tuned_f1.append(scores[selected_index])

    report = {
        "run_name": run_name,
        "checkpoint": audit["checkpoint"],
        "selected_epoch": audit["selected_epoch"],
        "checkpoint_sha256": audit["checkpoint_sha256"],
        "config_sha256": audit["config_sha256"],
        "prediction_path": str(prediction_path),
        "prediction_sha256": audit["prediction_sha256"],
        "fit_split": "valid",
        "samples": len(filenames),
        "attribute_names": attribute_names,
        "thresholds": thresholds,
        "objective": "per_attribute_F1",
        "grid": grid,
        "tie_break": "closest_to_0.5_then_lower",
        "comparison": ">=",
        "baseline_macro_f1": baseline["macro_f1"],
        "tuned_macro_f1": float(np.mean(tuned_f1)),
        "map": baseline["map"],
        "per_attribute_f1_at_0.5": baseline["per_attribute_f1"],
        "per_attribute_f1_tuned": tuned_f1,
        "per_attribute_ap": baseline["per_attribute_ap"],
        "source_sha256": config["source_sha256"],
        "environment": config["environment"],
        "test_evaluated": False,
        "note": (
            "Thresholds fitted on clean validation only. "
            "Tuned validation F1 is an in-sample fitting result, "
            "not independent evaluation. Reuse these thresholds "
            "unchanged for all clean and corrupted test conditions."
        ),
    }

    destination = (
        Path(project) / "runs/benchmark_preparation"
        / f"seed{config['seed']}" / f"{run_name}.json"
    )
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists():
        assert json.loads(
            destination.read_text(encoding="utf-8")
        ) == report, "Existing threshold report differs; preserved"
    else:
        write_json_atomic(destination, report)

    print("Historical validation IDs, targets and attribute order: PASS")
    print(f"F1@0.5: {report['baseline_macro_f1']:.6f}")
    print(f"F1(clean-validation tuned): {report['tuned_macro_f1']:.6f}")
    print(f"mAP: {report['map']:.6f}")
    print("Frozen thresholds:", len(thresholds))
    print("Saved:", destination)
    print("Clean-validation threshold freeze: PASS")
    print("No test evaluation performed")
    return report


THRESHOLD_REPORT = freeze_clean_thresholds(
    config=CONFIG,
    project=PROJECT,
    metric_function=evaluate_attributes,
)

Historical validation IDs, targets and attribute order: PASS
F1@0.5: 0.739332
F1(clean-validation tuned): 0.762971
mAP: 0.819523
Frozen thresholds: 24
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/benchmark_preparation/seed1/r18_seed1.json
Clean-validation threshold freeze: PASS
No test evaluation performed


In [4]:
import copy
import gc
import importlib
import importlib.metadata
import os
import platform
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.torch_version import TorchVersion


def prepare_rgb_run(model_name, seed, project, data_root):
    assert model_name in ("r18", "r50")
    assert isinstance(seed, int) and seed >= 1

    project = Path(project)
    data_root = Path(data_root)
    run_name = f"{model_name}_seed{seed}"
    reference_name = f"{model_name}_seed0"
    reference_root = project / "runs" / reference_name
    run_root = project / "runs" / run_name
    workspace = Path("/content") / f"seed{seed}_workspaces" / run_name

    assert torch.cuda.is_available(), "A GPU runtime is required"
    preparation_path = data_root.parent / "preparation.json"
    assert preparation_path.is_file(), "Shared data preparation is missing"

    reference_path = reference_root / "config.json"
    reference = json.loads(reference_path.read_text(encoding="utf-8"))

    # Check every source before creating or updating the workspace.
    source_payloads = {}
    for relative, expected_hash in reference["source_sha256"].items():
        relative_path = Path(relative)
        assert not relative_path.is_absolute()
        assert ".." not in relative_path.parts
        source = reference_root / "source" / relative_path
        assert sha_file(source) == expected_hash, (
            f"Historical source differs: {relative}"
        )
        source_payloads[relative] = source.read_bytes()

    environment = {
        "python": platform.python_version(),
        "torch": str(torch.__version__),
        "torchvision": importlib.metadata.version("torchvision"),
        "numpy": str(np.__version__),
        "Pillow": importlib.metadata.version("Pillow"),
        "scikit-learn": importlib.metadata.version("scikit-learn"),
    }

    config = copy.deepcopy(reference)
    config.pop("git_commit", None)
    config.update({
        "run_name": run_name,
        "seed": seed,
        "environment": environment,
        "reference_run": reference_name,
        "reference_config_sha256": sha_file(reference_path),
        "reference_source_commit": reference["git_commit"],
        "selection_tie_rule": "earliest epoch",
        "data_preparation_sha256": sha_file(preparation_path),
        "runtime": {
            "cuda": torch.version.cuda,
            "cudnn": torch.backends.cudnn.version(),
            "gpu": torch.cuda.get_device_name(0),
        },
        "reference_environment": reference["environment"],
    })

    config_path = run_root / "config.json"
    if config_path.exists():
        assert json.loads(
            config_path.read_text(encoding="utf-8")
        ) == config, "Existing run configuration differs; preserved"

    destinations = {}
    for relative, data in source_payloads.items():
        destinations[workspace / relative] = data
        destinations[run_root / "source" / relative] = data

    for destination, data in destinations.items():
        if destination.exists():
            assert destination.read_bytes() == data, (
                f"Existing source differs; preserved: {destination}"
            )

    for destination, data in destinations.items():
        if not destination.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(data)

    if not config_path.exists():
        write_json_atomic(config_path, config)

    # Import this model's exact historical implementation.
    for package in ("datasets", "models", "evaluation"):
        initializer = workspace / package / "__init__.py"
        if not initializer.exists():
            initializer.write_text("", encoding="utf-8")
        for name in list(sys.modules):
            if name == package or name.startswith(package + "."):
                del sys.modules[name]

    os.chdir(workspace)
    sys.path.insert(0, str(workspace))
    importlib.invalidate_caches()

    dataset_module = importlib.import_module("datasets.celeba")
    model_module = importlib.import_module(f"models.resnet{model_name[1:]}")
    metric_module = importlib.import_module("evaluation.attributes")

    for module in (dataset_module, model_module, metric_module):
        assert Path(module.__file__).resolve().is_relative_to(
            workspace.resolve()
        ), f"Unexpected module source: {module.__file__}"

    train_dataset = dataset_module.CelebAAttributes(
        split="train", root=data_root
    )
    valid_dataset = dataset_module.CelebAAttributes(
        split="valid", root=data_root
    )
    assert len(train_dataset) == 162770
    assert len(valid_dataset) == 19867
    assert train_dataset.attribute_names == config["attribute_names"]
    assert valid_dataset.attribute_names == config["attribute_names"]

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    model_class = getattr(
        model_module,
        {"r18": "ResNet18Attributes", "r50": "ResNet50Attributes"}[model_name],
    )
    device = torch.device("cuda")
    model = model_class(
        num_attributes=24, pretrained=not bool(checkpoints)
    ).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )

    if checkpoints:
        checkpoint_path = checkpoints[-1]
        with torch.serialization.safe_globals([TorchVersion]):
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=True
            )
        assert checkpoint["config"] == config
        model.load_state_dict(checkpoint["model_state"], strict=True)
        optimizer.load_state_dict(checkpoint["optimizer_state"])
        next_epoch = checkpoint["epoch"] + 1
        del checkpoint
        print("Restored:", checkpoint_path.name)
    else:
        checkpoint_path = run_root / "epoch_00.pt"
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": 0,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": config,
            "history": [],
        }, temporary)
        temporary.replace(checkpoint_path)
        next_epoch = 1
        print("Fresh ImageNet initialization saved:", checkpoint_path.name)

    model.eval()
    images = torch.stack([valid_dataset[i][0] for i in range(4)])
    with torch.inference_mode():
        logits = model(images.to(device))
    assert logits.shape == (4, 24)
    assert torch.isfinite(logits).all()
    assert all(parameter.requires_grad for parameter in model.parameters())

    print("Run:", run_name)
    print("Dataset sizes:", len(train_dataset), len(valid_dataset))
    print("Forward shape:", tuple(logits.shape))
    print("Next epoch:", next_epoch)
    print("Historical source and RGB initialization: PASS")
    print("No optimizer updates or test evaluation performed")

    return {
        "config": config,
        "model": model,
        "optimizer": optimizer,
        "criterion": nn.BCEWithLogitsLoss(),
        "train_dataset": train_dataset,
        "valid_dataset": valid_dataset,
        "device": device,
        "metric_function": metric_module.evaluate_attributes,
    }


# Release previous model references if they exist.
for variable_name in ("MODEL", "OPTIMIZER", "CRITERION", "RGB_CONTEXT"):
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()

RGB_CONTEXT = prepare_rgb_run(
    model_name="r50",
    seed=SEED,
    project=PROJECT,
    data_root=DATA_ROOT,
)

NameError: name 'DATA_ROOT' is not defined

In [ ]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def train_rgb(context, project):
    model = context["model"]
    optimizer = context["optimizer"]
    criterion = context["criterion"]
    train_dataset = context["train_dataset"]
    valid_dataset = context["valid_dataset"]
    config = context["config"]
    device = context["device"]
    metric_function = context["metric_function"]

    run_name = config["run_name"]
    model_name = run_name.split("_seed")[0]
    assert model_name in ("r18", "r50")
    assert run_name == f"{model_name}_seed{config['seed']}"
    assert torch.cuda.is_available()

    run_root = Path(project) / "runs" / run_name
    assert json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    ) == config, "Run configuration mismatch"

    for relative, expected_hash in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected_hash, (
            f"Archived source differs: {relative}"
        )

    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def make_loader(dataset, shuffle, epoch_seed):
        options = {
            "batch_size": config["batch_size"],
            "shuffle": shuffle,
            "num_workers": config["num_workers"],
            "worker_init_fn": seed_worker,
            "pin_memory": True,
            "drop_last": False,
        }
        # Preserve each model's historical DataLoader configuration.
        if shuffle or model_name == "r18":
            options["generator"] = (
                torch.Generator().manual_seed(epoch_seed)
            )
        return DataLoader(dataset, **options)

    def save_summaries(history):
        write_json_atomic(run_root / "history.json", history)
        if not history:
            return None
        best = max(history, key=lambda row: row["validation"]["map"])
        summary = {
            "epoch": best["epoch"],
            "checkpoint": f"epoch_{best['epoch']:02d}.pt",
            "validation_map": best["validation"]["map"],
            "validation_macro_f1": best["validation"]["macro_f1"],
        }
        write_json_atomic(run_root / "best.json", summary)
        return summary

    # Reload the latest completed epoch on every invocation.
    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Initialization checkpoint is missing"
    latest = checkpoints[-1]

    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )
    assert checkpoint["config"] == config
    model.load_state_dict(checkpoint["model_state"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    history = checkpoint["history"]
    completed_epoch = checkpoint["epoch"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    ), "Checkpoint history is not continuous"
    assert 0 <= completed_epoch <= config["epochs"]
    del checkpoint

    config["environment"]["torch"] = str(
        config["environment"]["torch"]
    )
    save_summaries(history)
    print("Run:", run_name, flush=True)
    print("Restored:", latest.name, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        train_loader = make_loader(train_dataset, True, epoch_seed)
        valid_loader = make_loader(valid_dataset, False, epoch_seed)

        torch.cuda.synchronize()
        started = time.perf_counter()
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        progress = tqdm(
            train_loader,
            desc=f"{run_name} epoch {epoch:02d} train",
        )
        for images, targets, _, _ in progress:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            assert torch.isfinite(loss), "Non-finite training loss"
            loss.backward()
            optimizer.step()

            count = images.size(0)
            train_loss_sum += loss.item() * count
            train_count += count

            if train_count % (config["batch_size"] * 50) == 0:
                progress.set_postfix(
                    BCE=f"{train_loss_sum / train_count:.6f}"
                )

        assert train_count == len(train_dataset)

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        logits_parts = []
        target_parts = []

        with torch.inference_mode():
            for images, targets, _, _ in tqdm(
                valid_loader,
                desc=f"{run_name} epoch {epoch:02d} valid",
            ):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                logits = model(images)
                loss = criterion(logits, targets)

                assert torch.isfinite(logits).all()
                assert torch.isfinite(loss), "Non-finite validation loss"

                count = images.size(0)
                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu())
                target_parts.append(targets.cpu())

        assert valid_count == len(valid_dataset)
        metrics = metric_function(
            torch.cat(logits_parts).numpy(),
            torch.cat(target_parts).numpy(),
            threshold=config["f1_threshold"],
        )
        del logits_parts, target_parts

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_loss": train_loss_sum / train_count,
            "valid_loss": valid_loss_sum / valid_count,
            "validation": metrics,
            "seconds_excluding_save": time.perf_counter() - started,
        }
        history.append(record)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": config,
            "history": history,
        }, temporary)
        temporary.replace(checkpoint_path)
        best = save_summaries(history)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_loss']:.6f} | "
            f"valid BCE={record['valid_loss']:.6f} | "
            f"F1@0.5={metrics['macro_f1']:.6f} | "
            f"mAP={metrics['map']:.6f} | "
            f"{record['seconds_excluding_save'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )

    best = save_summaries(history)
    print("Selected checkpoint:", run_root / best["checkpoint"])
    print("RGB training: COMPLETE")
    print("Validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best


RGB_HISTORY, RGB_BEST = train_rgb(
    context=RGB_CONTEXT,
    project=PROJECT,
)

Run: r50_seed1
Restored: epoch_00.pt
Next epoch: 1


r50_seed1 epoch 01 train:   0%|          | 0/2544 [00:00<?, ?it/s]

r50_seed1 epoch 01 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 01 saved | train BCE=0.210122 | valid BCE=0.196117 | F1@0.5=0.727617 | mAP=0.812597 | 31.3 min | best epoch=1


r50_seed1 epoch 02 train:   0%|          | 0/2544 [00:00<?, ?it/s]

...

Epoch 05 saved | train BCE=0.133943 | valid BCE=0.216093 | F1@0.5=0.743307 | mAP=0.806751 | 31.3 min | best epoch=3


r50_seed1 epoch 06 train:   0%|          | 0/2544 [00:00<?, ?it/s]

r50_seed1 epoch 06 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 06 saved | train BCE=0.109018 | valid BCE=0.245874 | F1@0.5=0.727868 | mAP=0.796718 | 31.2 min | best epoch=3


r50_seed1 epoch 07 train:   0%|          | 0/2544 [00:00<?, ?it/s]

In [1]:
from google.colab import drive
from pathlib import Path
import torch
from torch.torch_version import TorchVersion

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_ROOT = PROJECT / "runs/r50_seed1"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

checkpoints = sorted(RUN_ROOT.glob("epoch_[0-9][0-9].pt"))
assert checkpoints, "No saved checkpoints found"

print("Saved checkpoints:")
for path in checkpoints:
    print(path.name, "| bytes:", path.stat().st_size)

latest = checkpoints[-1]
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        latest, map_location="cpu", weights_only=True
    )

assert checkpoint["config"]["run_name"] == "r50_seed1"
assert checkpoint["config"]["seed"] == 1
assert [row["epoch"] for row in checkpoint["history"]] == list(
    range(1, checkpoint["epoch"] + 1)
)
assert "model_state" in checkpoint
assert "optimizer_state" in checkpoint

print("Latest checkpoint verified:", latest.name)
print("Completed epochs:", checkpoint["epoch"])
print("Next epoch:", checkpoint["epoch"] + 1)
print(
    "Local training images available:",
    Path("/content/seed1_shared_data/raw/img_align_celeba").is_dir(),
)
print("RGB context available:", "RGB_CONTEXT" in globals())
print("Training function available:", "train_rgb" in globals())
print("Checkpoint inspection: PASS")
del checkpoint

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
Saved checkpoints:
epoch_00.pt | bytes: 94555479
epoch_01.pt | bytes: 283151517
epoch_02.pt | bytes: 283152157
epoch_03.pt | bytes: 283152797
epoch_04.pt | bytes: 283153437
epoch_05.pt | bytes: 283154077
epoch_06.pt | bytes: 283154781
Latest checkpoint verified: epoch_06.pt
Completed epochs: 6
Next epoch: 7
Local training images available: False
RGB context available: False
Training function available: False
Checkpoint inspection: PASS


In [3]:
import json
from pathlib import Path


def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(
        json.dumps(payload, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


print("JSON checkpoint helpers restored")

JSON checkpoint helpers restored


In [5]:
from google.colab import drive
from pathlib import Path
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import random
import shutil
import sys
import zipfile

import numpy as np
import torch
from torch import nn
from torch.torch_version import TorchVersion

SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
DATA_ROOT = Path("/content/seed1_shared_data/raw")
WORKSPACE = Path("/content/seed1_workspaces/r50_seed1")
RUN_ROOT = PROJECT / "runs" / f"r50_seed{SEED}"

drive.mount("/content/drive")
assert torch.cuda.is_available(), "A GPU runtime is required"


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(
        json.dumps(payload, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def copy_verified(source, destination):
    source = Path(source)
    destination = Path(destination)
    expected_hash = sha_file(source)

    if destination.exists():
        assert sha_file(destination) == expected_hash, (
            f"Existing file differs; preserved: {destination}"
        )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + ".partial")
        shutil.copyfile(source, temporary)
        assert sha_file(temporary) == expected_hash
        temporary.replace(destination)

    return expected_hash


# Check the original run configuration before restoring anything.
CONFIG = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
assert CONFIG["run_name"] == f"r50_seed{SEED}"
assert CONFIG["seed"] == SEED

current_environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": importlib.metadata.version("torchvision"),
    "numpy": str(np.__version__),
    "Pillow": importlib.metadata.version("Pillow"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}
assert current_environment == CONFIG["environment"], (
    f"Environment changed. Expected: {CONFIG['environment']}; "
    f"current: {current_environment}"
)
assert {
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
} == CONFIG["runtime"], "CUDA, cuDNN or GPU differs from this run"

checkpoints = sorted(RUN_ROOT.glob("epoch_[0-9][0-9].pt"))
assert checkpoints, "Saved checkpoint is missing"
latest = checkpoints[-1]

with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        latest, map_location="cpu", weights_only=True
    )
assert checkpoint["config"] == CONFIG
assert [row["epoch"] for row in checkpoint["history"]] == list(
    range(1, checkpoint["epoch"] + 1)
)
print("Checkpoint verified:", latest.name)
print("Next epoch:", checkpoint["epoch"] + 1)

# Restore the shared annotations and original image archive.
DATA_ROOT.mkdir(parents=True, exist_ok=True)
annotation_hashes = {}

for filename in (
    "list_attr_celeba.txt",
    "list_eval_partition.txt",
    "list_landmarks_align_celeba.txt",
):
    annotation_hashes[filename] = copy_verified(
        PROJECT / "data" / filename,
        DATA_ROOT / filename,
    )

partitions = {"0": [], "1": [], "2": []}
with (DATA_ROOT / "list_eval_partition.txt").open(
    encoding="utf-8-sig"
) as stream:
    for line in stream:
        filename, split_code = line.split()
        partitions[split_code].append(filename)

assert {key: len(value) for key, value in partitions.items()} == {
    "0": 162770, "1": 19867, "2": 19962,
}
all_names = [name for names in partitions.values() for name in names]
assert len(all_names) == len(set(all_names))
required_names = partitions["0"] + partitions["1"]

local_zip = DATA_ROOT.parent / "img_align_celeba.zip"
print("Restoring shared image archive...", flush=True)
image_zip_hash = copy_verified(
    PROJECT / "data/img_align_celeba.zip",
    local_zip,
)
assert image_zip_hash == (
    "46fb89443c578308acf364d7d379fe1b9efb"
    "793042c0af734b6112e4fd3a8c74"
)

# Reconstruct and verify the exact original data preparation record.
reference_hashes = {
    f"{name}_seed0": sha_file(
        PROJECT / "runs" / f"{name}_seed0" / "config.json"
    )
    for name in ("r18", "r50", "m3", "oracle")
}
preparation = {
    "image_zip_sha256": image_zip_hash,
    "annotation_sha256": annotation_hashes,
    "reference_config_sha256": reference_hashes,
    "train_samples": 162770,
    "validation_samples": 19867,
    "training_started": False,
    "test_evaluated": False,
}
preparation_bytes = (
    json.dumps(preparation, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
assert hashlib.sha256(preparation_bytes).hexdigest() == (
    CONFIG["data_preparation_sha256"]
), "Original data preparation binding differs"

image_directory = DATA_ROOT / "img_align_celeba"
image_directory.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(local_zip) as archive:
    members = {}
    for member in archive.infolist():
        if member.is_dir():
            continue
        filename = Path(member.filename).name
        if filename.endswith(".jpg"):
            assert filename not in members
            members[filename] = member

    assert set(required_names).issubset(members)
    for index, filename in enumerate(required_names, start=1):
        data = archive.read(members[filename])
        destination = image_directory / filename
        if destination.exists():
            assert destination.read_bytes() == data, (
                f"Existing image differs: {filename}"
            )
        else:
            temporary = destination.with_name(filename + ".partial")
            temporary.write_bytes(data)
            temporary.replace(destination)

        if index % 20000 == 0 or index == len(required_names):
            print(
                f"Restored and verified images: {index}/{len(required_names)}",
                flush=True,
            )

preparation_path = DATA_ROOT.parent / "preparation.json"
if preparation_path.exists():
    assert preparation_path.read_bytes() == preparation_bytes
else:
    preparation_path.write_bytes(preparation_bytes)

# Restore the exact source archived by this Seed1 run.
for relative, expected_hash in CONFIG["source_sha256"].items():
    relative_path = Path(relative)
    assert not relative_path.is_absolute()
    assert ".." not in relative_path.parts

    source = RUN_ROOT / "source" / relative_path
    assert sha_file(source) == expected_hash, (
        f"Archived source differs: {relative}"
    )
    destination = WORKSPACE / relative_path
    if destination.exists():
        assert destination.read_bytes() == source.read_bytes()
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(source.read_bytes())

for package in ("datasets", "models", "evaluation"):
    initializer = WORKSPACE / package / "__init__.py"
    if not initializer.exists():
        initializer.write_text("", encoding="utf-8")
    for name in list(sys.modules):
        if name == package or name.startswith(package + "."):
            del sys.modules[name]

os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
importlib.invalidate_caches()

dataset_module = importlib.import_module("datasets.celeba")
model_module = importlib.import_module("models.resnet50")
metric_module = importlib.import_module("evaluation.attributes")

for module in (dataset_module, model_module, metric_module):
    assert Path(module.__file__).resolve().is_relative_to(
        WORKSPACE.resolve()
    )

train_dataset = dataset_module.CelebAAttributes(
    split="train", root=DATA_ROOT
)
valid_dataset = dataset_module.CelebAAttributes(
    split="valid", root=DATA_ROOT
)
assert train_dataset.names == partitions["0"]
assert valid_dataset.names == partitions["1"]
assert train_dataset.attribute_names == CONFIG["attribute_names"]
assert valid_dataset.attribute_names == CONFIG["attribute_names"]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = CONFIG["cudnn_deterministic"]
torch.backends.cudnn.benchmark = CONFIG["cudnn_benchmark"]

device = torch.device("cuda")

# Restore saved weights; do not download or initialize ImageNet weights.
model = model_module.ResNet50Attributes(
    num_attributes=24, pretrained=False
).to(device)
model.load_state_dict(checkpoint["model_state"], strict=True)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
optimizer.load_state_dict(checkpoint["optimizer_state"])

assert len(optimizer.state) > 0, "Adam state was not restored"
completed_epoch = checkpoint["epoch"]
del checkpoint

RGB_CONTEXT = {
    "config": CONFIG,
    "model": model,
    "optimizer": optimizer,
    "criterion": nn.BCEWithLogitsLoss(),
    "train_dataset": train_dataset,
    "valid_dataset": valid_dataset,
    "device": device,
    "metric_function": metric_module.evaluate_attributes,
}

print("Environment and original data binding: PASS")
print("Training/validation data and archived source: PASS")
print("Strict model loading and Adam restoration: PASS")
print("Completed epochs:", completed_epoch)
print("Next epoch:", completed_epoch + 1)
print("R50 Seed1 runtime restoration: PASS")
print("No optimizer updates or test evaluation performed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint verified: epoch_06.pt
Next epoch: 7
Restoring shared image archive...
Restored and verified images: 20000/182637
Restored and verified images: 40000/182637
Restored and verified images: 60000/182637
Restored and verified images: 80000/182637
Restored and verified images: 100000/182637
Restored and verified images: 120000/182637
Restored and verified images: 140000/182637
Restored and verified images: 160000/182637
Restored and verified images: 180000/182637
Restored and verified images: 182637/182637
Environment and original data binding: PASS
Training/validation data and archived source: PASS
Strict model loading and Adam restoration: PASS
Completed epochs: 6
Next epoch: 7
R50 Seed1 runtime restoration: PASS
No optimizer updates or test evaluation performed


In [1]:
import json
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def train_rgb(context, project):
    model = context["model"]
    optimizer = context["optimizer"]
    criterion = context["criterion"]
    train_dataset = context["train_dataset"]
    valid_dataset = context["valid_dataset"]
    config = context["config"]
    device = context["device"]
    metric_function = context["metric_function"]

    run_name = config["run_name"]
    model_name = run_name.split("_seed")[0]
    assert model_name in ("r18", "r50")
    assert run_name == f"{model_name}_seed{config['seed']}"
    assert torch.cuda.is_available()

    run_root = Path(project) / "runs" / run_name
    assert json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    ) == config, "Run configuration mismatch"

    for relative, expected_hash in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected_hash, (
            f"Archived source differs: {relative}"
        )

    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def make_loader(dataset, shuffle, epoch_seed):
        options = {
            "batch_size": config["batch_size"],
            "shuffle": shuffle,
            "num_workers": config["num_workers"],
            "worker_init_fn": seed_worker,
            "pin_memory": True,
            "drop_last": False,
        }
        # Preserve each model's historical DataLoader configuration.
        if shuffle or model_name == "r18":
            options["generator"] = (
                torch.Generator().manual_seed(epoch_seed)
            )
        return DataLoader(dataset, **options)

    def save_summaries(history):
        write_json_atomic(run_root / "history.json", history)
        if not history:
            return None

        # Equal mAP values retain the earliest epoch.
        best = max(history, key=lambda row: row["validation"]["map"])
        summary = {
            "epoch": best["epoch"],
            "checkpoint": f"epoch_{best['epoch']:02d}.pt",
            "validation_map": best["validation"]["map"],
            "validation_macro_f1": best["validation"]["macro_f1"],
        }
        write_json_atomic(run_root / "best.json", summary)
        return summary

    # Reload the latest completed epoch on every invocation.
    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Saved checkpoint is missing"
    latest = checkpoints[-1]

    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )

    assert checkpoint["config"] == config
    model.load_state_dict(checkpoint["model_state"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state"])

    history = checkpoint["history"]
    completed_epoch = checkpoint["epoch"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    ), "Checkpoint history is not continuous"
    assert 0 <= completed_epoch <= config["epochs"]
    del checkpoint

    config["environment"]["torch"] = str(
        config["environment"]["torch"]
    )
    save_summaries(history)

    print("Run:", run_name, flush=True)
    print("Restored:", latest.name, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        train_loader = make_loader(train_dataset, True, epoch_seed)
        valid_loader = make_loader(valid_dataset, False, epoch_seed)

        torch.cuda.synchronize()
        started = time.perf_counter()
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        progress = tqdm(
            train_loader,
            desc=f"{run_name} epoch {epoch:02d} train",
        )
        for images, targets, _, _ in progress:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            assert torch.isfinite(loss), "Non-finite training loss"

            loss.backward()
            optimizer.step()

            count = images.size(0)
            train_loss_sum += loss.item() * count
            train_count += count

            if train_count % (config["batch_size"] * 50) == 0:
                progress.set_postfix(
                    BCE=f"{train_loss_sum / train_count:.6f}"
                )

        assert train_count == len(train_dataset)

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        logits_parts = []
        target_parts = []

        with torch.inference_mode():
            for images, targets, _, _ in tqdm(
                valid_loader,
                desc=f"{run_name} epoch {epoch:02d} valid",
            ):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                logits = model(images)
                loss = criterion(logits, targets)

                assert torch.isfinite(logits).all()
                assert torch.isfinite(loss), "Non-finite validation loss"

                count = images.size(0)
                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu())
                target_parts.append(targets.cpu())

        assert valid_count == len(valid_dataset)

        metrics = metric_function(
            torch.cat(logits_parts).numpy(),
            torch.cat(target_parts).numpy(),
            threshold=config["f1_threshold"],
        )
        del logits_parts, target_parts

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_loss": train_loss_sum / train_count,
            "valid_loss": valid_loss_sum / valid_count,
            "validation": metrics,
            "seconds_excluding_save": time.perf_counter() - started,
        }
        history.append(record)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "config": config,
            "history": history,
        }, temporary)
        temporary.replace(checkpoint_path)
        best = save_summaries(history)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_loss']:.6f} | "
            f"valid BCE={record['valid_loss']:.6f} | "
            f"F1@0.5={metrics['macro_f1']:.6f} | "
            f"mAP={metrics['map']:.6f} | "
            f"{record['seconds_excluding_save'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )

    best = save_summaries(history)
    print("Selected checkpoint:", run_root / best["checkpoint"])
    print("RGB training: COMPLETE")
    print("Validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best


RGB_HISTORY, RGB_BEST = train_rgb(
    context=RGB_CONTEXT,
    project=PROJECT,
)

NameError: name 'RGB_CONTEXT' is not defined

In [2]:
from google.colab import drive
from pathlib import Path
import json
import math
import torch
from torch.torch_version import TorchVersion

drive.mount("/content/drive")

SEED = 1
RUN_ROOT = (
    Path("/content/drive/MyDrive/FacialVisualProfile")
    / "runs" / f"r50_seed{SEED}"
)

config = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
assert config["run_name"] == f"r50_seed{SEED}"
assert config["seed"] == SEED

checkpoints = sorted(RUN_ROOT.glob("epoch_[0-9][0-9].pt"))
assert checkpoints, "No checkpoints found"

print("Saved checkpoints:", [path.name for path in checkpoints])
latest = checkpoints[-1]

with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        latest, map_location="cpu", weights_only=True
    )

assert checkpoint["config"] == config
assert "model_state" in checkpoint
assert "optimizer_state" in checkpoint

completed = checkpoint["epoch"]
history = checkpoint["history"]
assert latest.name == f"epoch_{completed:02d}.pt"
assert 0 <= completed <= config["epochs"]
assert [row["epoch"] for row in history] == list(
    range(1, completed + 1)
)

for row in history:
    for value in (
        row["train_loss"],
        row["valid_loss"],
        row["validation"]["macro_f1"],
        row["validation"]["map"],
    ):
        assert math.isfinite(value), "Non-finite historical metric"

print("\nEpoch   Train BCE   Valid BCE   F1@0.5     mAP")
for row in history:
    print(
        f"{row['epoch']:>5}   "
        f"{row['train_loss']:.6f}    "
        f"{row['valid_loss']:.6f}    "
        f"{row['validation']['macro_f1']:.6f}   "
        f"{row['validation']['map']:.6f}"
    )

best = None
if history:
    best_row = max(history, key=lambda row: row["validation"]["map"])
    best = {
        "epoch": best_row["epoch"],
        "checkpoint": f"epoch_{best_row['epoch']:02d}.pt",
        "validation_map": best_row["validation"]["map"],
        "validation_macro_f1": best_row["validation"]["macro_f1"],
    }

# Check JSON summaries against the authoritative checkpoint history.
for filename, expected in (
    ("history.json", history),
    ("best.json", best),
):
    path = RUN_ROOT / filename
    if not path.exists():
        print(f"{filename}: MISSING; can rebuild from checkpoint")
    else:
        actual = json.loads(path.read_text(encoding="utf-8"))
        print(
            f"{filename}:",
            "MATCH" if actual == expected else "DIFFERS; requires review",
        )

del checkpoint

if best is not None:
    selected_path = RUN_ROOT / best["checkpoint"]
    assert selected_path.is_file(), "Selected checkpoint is missing"

    with torch.serialization.safe_globals([TorchVersion]):
        selected_checkpoint = torch.load(
            selected_path, map_location="cpu", weights_only=True
        )
    assert selected_checkpoint["config"] == config
    assert selected_checkpoint["epoch"] == best["epoch"]
    assert selected_checkpoint["history"] == history[:best["epoch"]]
    assert "model_state" in selected_checkpoint
    del selected_checkpoint

    print("\nSelected checkpoint verified:", best["checkpoint"])
    print(f"Selected F1@0.5: {best['validation_macro_f1']:.6f}")
    print(f"Selected mAP: {best['validation_map']:.6f}")

print("\nCompleted epochs:", completed, "/", config["epochs"])

if completed == config["epochs"]:
    assert all(
        (RUN_ROOT / f"epoch_{epoch:02d}.pt").is_file()
        for epoch in range(1, config["epochs"] + 1)
    ), "An epoch checkpoint is missing"
    print("R50 Seed1 training completion: PASS")
    print("No further training is needed")
else:
    print("R50 Seed1 training is incomplete")
    print("Resume from epoch:", completed + 1)

print(
    "Validation predictions exist:",
    (RUN_ROOT / "validation_predictions.npz").is_file(),
)
print("Read-only inspection completed; no training or files modified")

Mounted at /content/drive
Saved checkpoints: ['epoch_00.pt', 'epoch_01.pt', 'epoch_02.pt', 'epoch_03.pt', 'epoch_04.pt', 'epoch_05.pt', 'epoch_06.pt', 'epoch_07.pt', 'epoch_08.pt', 'epoch_09.pt', 'epoch_10.pt']

Epoch   Train BCE   Valid BCE   F1@0.5     mAP
    1   0.210122    0.196117    0.727617   0.812597
    2   0.185678    0.189363    0.722368   0.819725
    3   0.172438    0.191871    0.740484   0.821550
    4   0.155855    0.196002    0.744290   0.816426
    5   0.133943    0.216093    0.743307   0.806751
    6   0.109018    0.245874    0.727868   0.796718
    7   0.085994    0.264370    0.721055   0.790264
    8   0.067671    0.305702    0.720720   0.779533
    9   0.055069    0.327432    0.723199   0.784271
   10   0.046307    0.356513    0.716654   0.779750
history.json: MATCH
best.json: MATCH

Selected checkpoint verified: epoch_03.pt
Selected F1@0.5: 0.740484
Selected mAP: 0.821550

Completed epochs: 10 / 10
R50 Seed1 training completion: PASS
No further training is needed

In [3]:
from google.colab import drive
from pathlib import Path
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import random
import shutil
import sys
import zipfile

import numpy as np
import torch
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_ROOT = PROJECT / "runs" / f"r50_seed{SEED}"
DATA_ROOT = Path("/content/r50_validation_data/raw")
WORKSPACE = Path("/content/r50_validation_workspace")

drive.mount("/content/drive")
assert torch.cuda.is_available(), "Use the T4 GPU runtime"


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def copy_verified(source, destination):
    source, destination = Path(source), Path(destination)
    expected = sha_file(source)
    if destination.exists():
        assert sha_file(destination) == expected, (
            f"Existing file differs; preserved: {destination}"
        )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + ".partial")
        shutil.copyfile(source, temporary)
        assert sha_file(temporary) == expected
        temporary.replace(destination)
    return expected


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


config = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
best = json.loads(
    (RUN_ROOT / "best.json").read_text(encoding="utf-8")
)
assert config["run_name"] == f"r50_seed{SEED}"
assert config["seed"] == SEED

environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": importlib.metadata.version("torchvision"),
    "numpy": str(np.__version__),
    "Pillow": importlib.metadata.version("Pillow"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}
assert environment == config["environment"], "Inference environment differs"
assert {
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
} == config["runtime"], "CUDA, cuDNN or GPU differs"

# Verify the frozen training data binding before using validation images.
DATA_ROOT.mkdir(parents=True, exist_ok=True)
annotation_hashes = {}
for filename in (
    "list_attr_celeba.txt",
    "list_eval_partition.txt",
    "list_landmarks_align_celeba.txt",
):
    annotation_hashes[filename] = copy_verified(
        PROJECT / "data" / filename, DATA_ROOT / filename
    )

preparation = {
    "image_zip_sha256": (
        "46fb89443c578308acf364d7d379fe1b9efb"
        "793042c0af734b6112e4fd3a8c74"
    ),
    "annotation_sha256": annotation_hashes,
    "reference_config_sha256": {
        f"{name}_seed0": sha_file(
            PROJECT / "runs" / f"{name}_seed0/config.json"
        )
        for name in ("r18", "r50", "m3", "oracle")
    },
    "train_samples": 162770,
    "validation_samples": 19867,
    "training_started": False,
    "test_evaluated": False,
}
preparation_bytes = (
    json.dumps(preparation, indent=2, allow_nan=False) + "\n"
).encode("utf-8")
assert hashlib.sha256(preparation_bytes).hexdigest() == (
    config["data_preparation_sha256"]
), "Original data binding differs"

with (DATA_ROOT / "list_eval_partition.txt").open(
    encoding="utf-8-sig"
) as stream:
    validation_names = [
        name for line in stream
        for name, split in [line.split()]
        if split == "1"
    ]
assert len(validation_names) == len(set(validation_names)) == 19867

# Reuse the local archive if it survived; otherwise restore it from Drive.
archive_candidates = (
    Path("/content/seed1_shared_data/img_align_celeba.zip"),
    DATA_ROOT.parent / "img_align_celeba.zip",
)
local_zip = next(
    (path for path in archive_candidates if path.is_file()),
    DATA_ROOT.parent / "img_align_celeba.zip",
)
if not local_zip.exists():
    print("Restoring image archive...", flush=True)
    copy_verified(PROJECT / "data/img_align_celeba.zip", local_zip)

assert sha_file(local_zip) == preparation["image_zip_sha256"]

image_directory = DATA_ROOT / "img_align_celeba"
image_directory.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(local_zip) as archive:
    members = {}
    for member in archive.infolist():
        if member.is_dir():
            continue
        name = Path(member.filename).name
        if name.endswith(".jpg"):
            assert name not in members
            members[name] = member

    for index, name in enumerate(validation_names, start=1):
        data = archive.read(members[name])
        destination = image_directory / name
        if destination.exists():
            assert destination.read_bytes() == data
        else:
            temporary = destination.with_name(name + ".partial")
            temporary.write_bytes(data)
            temporary.replace(destination)

        if index % 5000 == 0 or index == len(validation_names):
            print(f"Verified validation images: {index}/19867", flush=True)

# Import the exact source archived by this run.
for relative, expected in config["source_sha256"].items():
    source = RUN_ROOT / "source" / relative
    assert sha_file(source) == expected, f"Source differs: {relative}"
    destination = WORKSPACE / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        assert destination.read_bytes() == source.read_bytes()
    else:
        destination.write_bytes(source.read_bytes())

for package in ("datasets", "models", "evaluation"):
    initializer = WORKSPACE / package / "__init__.py"
    if not initializer.exists():
        initializer.write_text("", encoding="utf-8")
    for name in list(sys.modules):
        if name == package or name.startswith(package + "."):
            del sys.modules[name]

os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))
importlib.invalidate_caches()

dataset_module = importlib.import_module("datasets.celeba")
model_module = importlib.import_module("models.resnet50")
metric_module = importlib.import_module("evaluation.attributes")

for module in (dataset_module, model_module, metric_module):
    assert Path(module.__file__).resolve().is_relative_to(
        WORKSPACE.resolve()
    )

dataset = dataset_module.CelebAAttributes(
    split="valid", root=DATA_ROOT
)
assert dataset.names == validation_names
assert dataset.attribute_names == config["attribute_names"]

checkpoint_path = RUN_ROOT / best["checkpoint"]
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )
assert checkpoint["config"] == config
assert checkpoint["epoch"] == best["epoch"]

device = torch.device("cuda")
model = model_module.ResNet50Attributes(
    num_attributes=24, pretrained=False
).to(device)
model.load_state_dict(checkpoint["model_state"], strict=True)
del checkpoint

torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
torch.backends.cudnn.benchmark = config["cudnn_benchmark"]
model.eval()

loader = DataLoader(
    dataset,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"],
    worker_init_fn=seed_worker,
    pin_memory=True,
    drop_last=False,
)

logits_parts, target_parts, filenames = [], [], []
with torch.inference_mode():
    for images, targets, _, names in tqdm(
        loader, desc=f"{config['run_name']} selected validation"
    ):
        logits = model(images.to(device, non_blocking=True))
        assert torch.isfinite(logits).all()
        logits_parts.append(logits.cpu().numpy())
        target_parts.append(targets.numpy())
        filenames.extend(names)

arrays = {
    "filenames": np.asarray(filenames),
    "logits": np.concatenate(logits_parts),
    "targets": np.concatenate(target_parts),
    "attribute_names": np.asarray(dataset.attribute_names),
}
assert filenames == validation_names
assert arrays["logits"].shape == arrays["targets"].shape == (19867, 24)
assert np.array_equal(
    arrays["targets"],
    torch.stack([dataset.targets[name] for name in filenames]).numpy(),
)

metrics = metric_module.evaluate_attributes(
    arrays["logits"], arrays["targets"], threshold=0.5
)
assert abs(
    metrics["macro_f1"] - best["validation_macro_f1"]
) <= 1e-10, "F1 reproduction failed"
assert abs(
    metrics["map"] - best["validation_map"]
) <= 1e-10, "mAP reproduction failed"

prediction_path = RUN_ROOT / "validation_predictions.npz"
if prediction_path.exists():
    with np.load(prediction_path, allow_pickle=False) as existing:
        assert set(existing.files) == set(arrays)
        for key, value in arrays.items():
            assert np.array_equal(existing[key], value), (
                f"Existing predictions differ; preserved: {key}"
            )
else:
    temporary = prediction_path.with_name(prediction_path.name + ".partial")
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, **arrays)
    temporary.replace(prediction_path)

report = {
    "run_name": config["run_name"],
    "selected_epoch": best["epoch"],
    "checkpoint": best["checkpoint"],
    "checkpoint_sha256": sha_file(checkpoint_path),
    "config_sha256": sha_file(RUN_ROOT / "config.json"),
    "prediction_sha256": sha_file(prediction_path),
    "split": "valid",
    "samples": len(filenames),
    "attribute_names": config["attribute_names"],
    "metrics": metrics,
    "threshold_fitted": False,
    "test_evaluated": False,
    "passed": True,
}
report_path = RUN_ROOT / "validation_export_audit.json"
if report_path.exists():
    assert json.loads(report_path.read_text(encoding="utf-8")) == report
else:
    temporary = report_path.with_name(report_path.name + ".partial")
    temporary.write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(report_path)

print("Full validation IDs, targets and attribute order: PASS")
print("Selected-checkpoint metric reproduction: PASS")
print(f"F1@0.5: {metrics['macro_f1']:.6f}")
print(f"mAP: {metrics['map']:.6f}")
print("Saved:", prediction_path)
print("R50 Seed1 validation export: PASS")
print("No training, threshold fitting or test evaluation performed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restoring image archive...
Verified validation images: 5000/19867
Verified validation images: 10000/19867
Verified validation images: 15000/19867
Verified validation images: 19867/19867


r50_seed1 selected validation:   0%|          | 0/311 [00:00<?, ?it/s]

Full validation IDs, targets and attribute order: PASS
Selected-checkpoint metric reproduction: PASS
F1@0.5: 0.740484
mAP: 0.821550
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/r50_seed1/validation_predictions.npz
R50 Seed1 validation export: PASS
No training, threshold fitting or test evaluation performed


In [4]:
from google.colab import drive
from pathlib import Path
import hashlib
import json

import numpy as np
from sklearn.metrics import average_precision_score, f1_score

drive.mount("/content/drive")

# Change these parameters when reusing this cell.
SEED = 1
MODEL_NAME = "r50"
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")

RUN_NAME = f"{MODEL_NAME}_seed{SEED}"
RUN_ROOT = PROJECT / "runs" / RUN_NAME
PREDICTION_PATH = RUN_ROOT / "validation_predictions.npz"
REFERENCE_PATH = (
    PROJECT / "runs/r18_seed0/day7/validation_predictions.npz"
)
OUTPUT_PATH = (
    PROJECT / "runs/benchmark_preparation"
    / f"seed{SEED}" / f"{RUN_NAME}.json"
)


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


config = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
best = json.loads(
    (RUN_ROOT / "best.json").read_text(encoding="utf-8")
)

assert config["seed"] == SEED, "Seed mismatch"
assert config["run_name"] == RUN_NAME, "Run name mismatch"
assert config["selection_metric"] == "validation_mAP"
assert config["selection_tie_rule"] == "earliest epoch"

checkpoint_path = RUN_ROOT / best["checkpoint"]
assert checkpoint_path.is_file(), "Selected checkpoint is missing"
assert best["checkpoint"] == f"epoch_{best['epoch']:02d}.pt"

for relative, expected in config["source_sha256"].items():
    assert sha_file(RUN_ROOT / "source" / relative) == expected, (
        f"Archived source differs: {relative}"
    )

with np.load(PREDICTION_PATH, allow_pickle=False) as saved:
    filenames = saved["filenames"].copy()
    logits = saved["logits"].copy()
    targets = saved["targets"].copy()
    attribute_names = saved["attribute_names"].tolist()

assert logits.shape == targets.shape == (19867, 24)
assert filenames.shape == (19867,)
assert len(set(filenames.tolist())) == 19867
assert np.isfinite(logits).all(), "Non-finite logits"
assert np.isin(targets, [0, 1]).all(), "Non-binary targets"
assert attribute_names == config["attribute_names"]
assert np.all((targets.sum(axis=0) > 0) & (targets.sum(axis=0) < 19867))

with np.load(REFERENCE_PATH, allow_pickle=False) as reference:
    assert np.array_equal(filenames, reference["filenames"]), (
        "Historical validation IDs or order differ"
    )
    assert np.array_equal(targets, reference["targets"]), (
        "Historical validation targets differ"
    )
    assert attribute_names == reference["attribute_names"].tolist(), (
        "Historical attribute order differs"
    )

print("Historical validation IDs, targets and attribute order: PASS")

# Use a numerically stable float64 sigmoid.
values = logits.astype(np.float64)
probabilities = np.empty_like(values)
positive = values >= 0
probabilities[positive] = 1.0 / (1.0 + np.exp(-values[positive]))
exp_values = np.exp(values[~positive])
probabilities[~positive] = exp_values / (1.0 + exp_values)

grid_integer = list(range(5, 96))
grid = [value / 100 for value in grid_integer]

baseline_f1 = []
tuned_f1 = []
attribute_ap = []
thresholds = []

for column in range(24):
    labels = targets[:, column]
    scores = probabilities[:, column]

    baseline_f1.append(float(f1_score(
        labels, scores >= 0.5, zero_division=0
    )))
    attribute_ap.append(float(average_precision_score(labels, scores)))

    candidate_f1 = [
        float(f1_score(labels, scores >= threshold, zero_division=0))
        for threshold in grid
    ]

    # Resolve exact F1 ties by distance to 0.5, then lower threshold.
    selected = min(
        range(len(grid)),
        key=lambda index: (
            -candidate_f1[index],
            abs(grid_integer[index] - 50),
            grid_integer[index],
        ),
    )
    thresholds.append(grid[selected])
    tuned_f1.append(candidate_f1[selected])

baseline_macro_f1 = float(np.mean(baseline_f1))
tuned_macro_f1 = float(np.mean(tuned_f1))
mean_ap = float(np.mean(attribute_ap))

assert abs(baseline_macro_f1 - best["validation_macro_f1"]) <= 1e-10, (
    "Selected-checkpoint F1 reproduction failed"
)
assert abs(mean_ap - best["validation_map"]) <= 1e-10, (
    "Selected-checkpoint mAP reproduction failed"
)
assert all(tuned >= baseline for tuned, baseline in zip(
    tuned_f1, baseline_f1
)), "Threshold fitting reduced validation F1"

print("Selected-checkpoint metric reproduction: PASS")
print("Computing checkpoint and prediction SHA256...")

report = {
    "run_name": RUN_NAME,
    "checkpoint": best["checkpoint"],
    "selected_epoch": best["epoch"],
    "checkpoint_sha256": sha_file(checkpoint_path),
    "config_sha256": sha_file(RUN_ROOT / "config.json"),
    "prediction_path": str(PREDICTION_PATH),
    "prediction_sha256": sha_file(PREDICTION_PATH),
    "fit_split": "valid",
    "samples": 19867,
    "attribute_names": attribute_names,
    "thresholds": thresholds,
    "objective": "per_attribute_F1",
    "grid": grid,
    "tie_break": "closest_to_0.5_then_lower",
    "comparison": ">=",
    "baseline_macro_f1": baseline_macro_f1,
    "tuned_macro_f1": tuned_macro_f1,
    "map": mean_ap,
    "per_attribute_f1_at_0.5": baseline_f1,
    "per_attribute_f1_tuned": tuned_f1,
    "per_attribute_ap": attribute_ap,
    "source_sha256": config["source_sha256"],
    "environment": config["environment"],
    "test_evaluated": False,
    "note": (
        "Thresholds fitted on clean validation only. Tuned validation "
        "F1 is an in-sample fitting result, not independent evaluation. "
        "Reuse these thresholds unchanged for all clean and corrupted "
        "test conditions."
    ),
}

if OUTPUT_PATH.exists():
    existing = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))
    assert existing == report, (
        f"Existing threshold report differs; preserved: {OUTPUT_PATH}"
    )
else:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary = OUTPUT_PATH.with_suffix(".json.partial")
    temporary.write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(OUTPUT_PATH)

print("Run:", RUN_NAME)
print(f"F1@0.5: {baseline_macro_f1:.6f}")
print(f"F1(clean-validation tuned): {tuned_macro_f1:.6f}")
print(f"mAP: {mean_ap:.6f}")
print("Frozen thresholds:", len(thresholds))
print("Saved:", OUTPUT_PATH)
print("Clean-validation threshold freeze: PASS")
print("No model inference, training or test evaluation performed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Historical validation IDs, targets and attribute order: PASS
Selected-checkpoint metric reproduction: PASS
Computing checkpoint and prediction SHA256...
Run: r50_seed1
F1@0.5: 0.740484
F1(clean-validation tuned): 0.765545
mAP: 0.821550
Frozen thresholds: 24
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/benchmark_preparation/seed1/r50_seed1.json
Clean-validation threshold freeze: PASS
No model inference, training or test evaluation performed


In [5]:
from google.colab import drive
from pathlib import Path
from collections import Counter
import hashlib
import importlib
import json
import os
import shutil
import sqlite3
import sys
import zipfile

import torch

drive.mount("/content/drive")

# Change only this parameter for Seed2.
SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_NAME = f"m3_seed{SEED}"
REFERENCE_ROOT = PROJECT / "runs/m3_seed0"
RUN_ROOT = PROJECT / "runs" / RUN_NAME
WORKSPACE = Path("/content/multiseed_workspaces") / RUN_NAME
DATA_ROOT = Path("/content/seed1_shared_data/raw")
CACHE_PATH = Path("/content/multiseed_shared_cache/mtcnn_clean.sqlite")

assert SEED in (1, 2)
assert torch.cuda.is_available(), "A GPU runtime is required"
assert REFERENCE_ROOT.is_dir(), "Historical M3 run is missing"

reference = json.loads(
    (REFERENCE_ROOT / "config.json").read_text(encoding="utf-8")
)

assert reference["model"] == "ResNet18GeometryAttributes"
assert reference["failure_policy"] == (
    "retain sample; mask geometry features after MLP"
)
assert reference["selection_metric"] == "validation_mAP"
assert reference["selection_tie_rule"] == "earliest epoch"


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def copy_verified(source, destination):
    source = Path(source)
    destination = Path(destination)
    assert source.is_file(), f"Missing source: {source}"
    expected = sha_file(source)

    if destination.exists():
        assert sha_file(destination) == expected, (
            f"Existing file differs; preserved: {destination}"
        )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + ".partial")
        shutil.copyfile(source, temporary)
        assert sha_file(temporary) == expected
        temporary.replace(destination)

    return expected


# Reuse the exact historical source, including explanatory comments.
for relative, expected in reference["source_sha256"].items():
    source = REFERENCE_ROOT / "source" / relative
    assert sha_file(source) == expected, (
        f"Historical source hash mismatch: {relative}"
    )
    copy_verified(source, WORKSPACE / relative)

for package in ("datasets", "models", "evaluation"):
    initializer = WORKSPACE / package / "__init__.py"
    if not initializer.exists():
        initializer.write_text("", encoding="utf-8")

print("Historical M3 source snapshot: PASS")

annotation_hashes = {}
for filename in (
    "list_attr_celeba.txt",
    "list_eval_partition.txt",
    "list_landmarks_align_celeba.txt",
):
    annotation_hashes[filename] = copy_verified(
        PROJECT / "data" / filename,
        DATA_ROOT / filename,
    )

cache_hash = copy_verified(
    PROJECT / "cache/mtcnn_clean.sqlite",
    CACHE_PATH,
)

partition_path = DATA_ROOT / "list_eval_partition.txt"
rows = [
    line.split()
    for line in partition_path.read_text(
        encoding="utf-8-sig"
    ).splitlines()
]
assert all(
    len(row) == 2 and row[1] in ("0", "1", "2")
    for row in rows
)
partitions = dict(rows)
assert len(partitions) == len(rows) == 202599
assert Counter(partitions.values()) == {
    "0": 162770, "1": 19867, "2": 19962
}

with sqlite3.connect(
    CACHE_PATH.resolve().as_uri() + "?mode=ro", uri=True
) as connection:
    assert connection.execute("PRAGMA quick_check").fetchone()[0] == "ok"
    metadata = connection.execute(
        "SELECT value FROM metadata WHERE key = 'config'"
    ).fetchone()
    assert metadata is not None
    cache_config = json.loads(metadata[0])
    assert cache_config["partition_sha256"] == (
        annotation_hashes["list_eval_partition.txt"]
    )
    records = {
        name: (split, encoded)
        for name, split, encoded in connection.execute(
            "SELECT name, split, result FROM detections"
        )
    }

assert set(records) == set(partitions), "Cache membership mismatch"

expected_zip_hash = (
    "46fb89443c578308acf364d7d379fe1b9"
    "efb793042c0af734b6112e4fd3a8c74"
)
candidates = (
    Path("/content/seed1_shared_data/img_align_celeba.zip"),
    Path("/content/r50_validation_data/img_align_celeba.zip"),
)
archive_path = next(
    (path for path in candidates if path.is_file()), None
)
if archive_path is None:
    archive_path = candidates[0]
    print("Copying shared image archive...", flush=True)
    copy_verified(
        PROJECT / "data/img_align_celeba.zip",
        archive_path,
    )

assert sha_file(archive_path) == expected_zip_hash, (
    "Original image archive hash mismatch"
)

image_root = DATA_ROOT / "img_align_celeba"
image_root.mkdir(parents=True, exist_ok=True)
selected_rows = [
    (name, split) for name, split in rows if split in ("0", "1")
]
counts = Counter()

with zipfile.ZipFile(archive_path) as archive:
    members = {}
    for member in archive.namelist():
        if member.endswith("/"):
            continue
        assert member.lower().endswith(".jpg"), (
            f"Unexpected archive member: {member}"
        )
        name = Path(member).name
        assert name not in members, f"Duplicate image ID: {name}"
        members[name] = member

    assert set(members) == set(partitions)

    for index, (name, split) in enumerate(selected_rows, start=1):
        cached_split, encoded = records[name]
        assert cached_split == split, f"Cache split mismatch: {name}"
        result = json.loads(encoded)
        assert result["status"] in ("ok", "no_face", "invalid_output")
        counts[(split, result["status"])] += 1

        # ZIP reads check CRC; cached hashes bind the original image bytes.
        content = archive.read(members[name])
        expected = result["image_sha256"]
        assert hashlib.sha256(content).hexdigest() == expected, (
            f"Cached image hash mismatch: {name}"
        )

        destination = image_root / name
        if destination.exists():
            assert sha_file(destination) == expected, (
                f"Existing image differs; preserved: {name}"
            )
        else:
            destination.write_bytes(content)

        if index % 20000 == 0 or index == len(selected_rows):
            print(
                f"Verified train/validation images: "
                f"{index}/{len(selected_rows)}",
                flush=True,
            )

assert counts == Counter({
    ("0", "ok"): 162521,
    ("0", "no_face"): 249,
    ("1", "ok"): 19835,
    ("1", "no_face"): 32,
}), f"Historical cache status counts differ: {counts}"

# Clear prior model/dataset imports before loading this workspace.
for module_name in list(sys.modules):
    if any(
        module_name == package or module_name.startswith(package + ".")
        for package in ("datasets", "models", "evaluation")
    ):
        del sys.modules[module_name]

workspace_string = str(WORKSPACE)
if workspace_string in sys.path:
    sys.path.remove(workspace_string)
sys.path.insert(0, workspace_string)
os.chdir(WORKSPACE)
importlib.invalidate_caches()

from datasets.celeba_predicted import CelebAPredictedAttributes

TRAIN_DATASET = CelebAPredictedAttributes(
    CACHE_PATH,
    split="train",
    root=DATA_ROOT,
    failure_policy="mask",
)
VALID_DATASET = CelebAPredictedAttributes(
    CACHE_PATH,
    split="valid",
    root=DATA_ROOT,
    failure_policy="mask",
)

assert TRAIN_DATASET.names == [
    name for name, split in rows if split == "0"
]
assert VALID_DATASET.names == [
    name for name, split in rows if split == "1"
]
assert TRAIN_DATASET.attribute_names == reference["attribute_names"]
assert VALID_DATASET.attribute_names == reference["attribute_names"]

# Confirm that a real detection failure remains in the dataset.
failure_index = TRAIN_DATASET.names.index("000199.jpg")
image, targets, points, filename, geometry_valid = (
    TRAIN_DATASET[failure_index]
)
assert filename == "000199.jpg"
assert not geometry_valid
assert torch.equal(points, torch.zeros(10))
assert image.shape == (3, 224, 224)
assert targets.shape == (24,)
assert torch.isfinite(image).all()

M3_PREPARATION = {
    "run_name": RUN_NAME,
    "seed": SEED,
    "reference_config_sha256": sha_file(REFERENCE_ROOT / "config.json"),
    "source_sha256": reference["source_sha256"],
    "image_zip_sha256": expected_zip_hash,
    "annotation_sha256": annotation_hashes,
    "cache_sha256": cache_hash,
    "train_samples": len(TRAIN_DATASET),
    "validation_samples": len(VALID_DATASET),
    "masked_train_samples": 249,
    "masked_validation_samples": 32,
}

del records

print("GPU:", torch.cuda.get_device_name(0))
print("Run:", RUN_NAME)
print("Dataset sizes:", len(TRAIN_DATASET), len(VALID_DATASET))
print("Masked geometry samples: train=249; validation=32")
print("Historical failure sample retained with masked geometry: PASS")
print("Workspace:", WORKSPACE)
print("Data root:", DATA_ROOT)
print("Local landmark cache:", CACHE_PATH)
print("Existing run directory:", RUN_ROOT.exists())
print("M3 shared data and source preparation: PASS")
print("No model initialization, optimizer updates or test evaluation performed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Historical M3 source snapshot: PASS
Verified train/validation images: 20000/182637
Verified train/validation images: 40000/182637
Verified train/validation images: 60000/182637
Verified train/validation images: 80000/182637
Verified train/validation images: 100000/182637
Verified train/validation images: 120000/182637
Verified train/validation images: 140000/182637
Verified train/validation images: 160000/182637
Verified train/validation images: 180000/182637
Verified train/validation images: 182637/182637
GPU: Tesla T4
Run: m3_seed1
Dataset sizes: 162770 19867
Masked geometry samples: train=249; validation=32
Historical failure sample retained with masked geometry: PASS
Workspace: /content/multiseed_workspaces/m3_seed1
Data root: /content/seed1_shared_data/raw
Local landmark cache: /content/multiseed_shared_cache/mtcnn_clean.sqlite
Existing run directory: Fa

In [8]:
import copy
import importlib.metadata
import json
import platform
import random
import shutil

import numpy as np
import torch
from torch import nn
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader, Subset

from models.resnet18_geometry import ResNet18GeometryAttributes

required = (
    "SEED", "PROJECT", "RUN_NAME", "RUN_ROOT", "WORKSPACE",
    "TRAIN_DATASET", "VALID_DATASET", "M3_PREPARATION",
    "REFERENCE_ROOT", "sha_file",
)
missing = [name for name in required if name not in globals()]
assert not missing, f"Run the M3 preparation cell first: {missing}"


def write_json_atomic(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def set_seed(value):
    random.seed(value)
    np.random.seed(value)
    torch.manual_seed(value)
    torch.cuda.manual_seed_all(value)


environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": importlib.metadata.version("torchvision"),
    "numpy": importlib.metadata.version("numpy"),
    "Pillow": importlib.metadata.version("Pillow"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}
runtime = {
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
}

reference = json.loads(
    (REFERENCE_ROOT / "config.json").read_text(encoding="utf-8")
)

# Verify the environment used for the completed Seed1 RGB runs.
if SEED == 1:
    rgb_config = json.loads(
        (PROJECT / "runs/r50_seed1/config.json").read_text(
            encoding="utf-8"
        )
    )
    for key, value in environment.items():
        assert rgb_config["environment"][key] == value, (
            f"Seed1 environment mismatch: {key}"
        )
    assert rgb_config["runtime"] == runtime, (
        "Seed1 CUDA, cuDNN or GPU mismatch"
    )

CONFIG = copy.deepcopy(reference)
CONFIG.pop("git_commit", None)
CONFIG.update({
    "run_name": RUN_NAME,
    "seed": SEED,
    "environment": environment,
    "runtime": runtime,
    "reference_run": "m3_seed0",
    "reference_config_sha256": M3_PREPARATION[
        "reference_config_sha256"
    ],
    "reference_source_commit": reference["git_commit"],
    "reference_environment": reference["environment"],
    "data_preparation_sha256": __import__("hashlib").sha256(
        (
            json.dumps(M3_PREPARATION, indent=2, allow_nan=False)
            + "\n"
        ).encode("utf-8")
    ).hexdigest(),
    "cache_sha256": M3_PREPARATION["cache_sha256"],
})

RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Check existing records before writing.
for filename, value in (
    ("config.json", CONFIG),
    ("data_preparation.json", M3_PREPARATION),
):
    path = RUN_ROOT / filename
    if path.exists():
        assert json.loads(path.read_text(encoding="utf-8")) == value, (
            f"Existing record differs; preserved: {path}"
        )

for relative, expected in CONFIG["source_sha256"].items():
    source = WORKSPACE / relative
    destination = RUN_ROOT / "source" / relative
    assert sha_file(source) == expected
    if destination.exists():
        assert sha_file(destination) == expected, (
            f"Existing archived source differs: {relative}"
        )

for relative, expected in CONFIG["source_sha256"].items():
    destination = RUN_ROOT / "source" / relative
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(WORKSPACE / relative, destination)
    assert sha_file(destination) == expected

for filename, value in (
    ("config.json", CONFIG),
    ("data_preparation.json", M3_PREPARATION),
):
    if not (RUN_ROOT / filename).exists():
        write_json_atomic(RUN_ROOT / filename, value)

DEVICE = torch.device("cuda")
torch.backends.cudnn.deterministic = CONFIG["cudnn_deterministic"]
torch.backends.cudnn.benchmark = CONFIG["cudnn_benchmark"]

checkpoints = sorted(RUN_ROOT.glob("epoch_[0-9][0-9].pt"))
set_seed(SEED)

MODEL = ResNet18GeometryAttributes(
    pretrained=not bool(checkpoints)
).to(DEVICE)
OPTIMIZER = torch.optim.Adam(
    MODEL.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
CRITERION = nn.BCEWithLogitsLoss()

assert all(parameter.requires_grad for parameter in MODEL.parameters())
assert sum(parameter.numel() for parameter in MODEL.parameters()) == (
    11_191_000
)

if checkpoints:
    latest = checkpoints[-1]
    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )

    assert checkpoint["config"] == CONFIG
    completed_epoch = checkpoint["epoch"]
    assert latest.name == f"epoch_{completed_epoch:02d}.pt"
    assert 0 <= completed_epoch <= CONFIG["epochs"]
    assert [row["epoch"] for row in checkpoint["history"]] == list(
        range(1, completed_epoch + 1)
    )

    MODEL.load_state_dict(
        checkpoint["model_state_dict"], strict=True
    )
    OPTIMIZER.load_state_dict(checkpoint["optimizer_state_dict"])
    del checkpoint
    print("Restored:", latest.name)
else:
    completed_epoch = 0

# Include a real detection failure in the historical batch-64 smoke check.
failure_index = TRAIN_DATASET.names.index("000199.jpg")
indices = list(range(63)) + [failure_index]
assert len(set(indices)) == 64

smoke_loader = DataLoader(
    Subset(TRAIN_DATASET, indices),
    batch_size=64,
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)
images, targets, points, filenames, valid = next(iter(smoke_loader))
assert not valid[-1].item()

images = images.to(DEVICE)
targets = targets.to(DEVICE)
points = points.to(DEVICE)
valid = valid.to(DEVICE)

MODEL.eval()
with torch.inference_mode():
    logits = MODEL(images, points, valid)
    loss = CRITERION(logits, targets)

    assert logits.shape == (64, 24)
    assert torch.isfinite(logits).all()
    assert torch.isfinite(loss)

    invariance = {}
    for label, replacement in (
        ("large_finite", 1e6),
        ("nan", float("nan")),
    ):
        altered = points.clone()
        altered[~valid] = replacement
        altered_logits = MODEL(images, altered, valid)
        difference = float((altered_logits - logits).abs().max().item())
        assert difference == 0.0, (
            f"Invalid-coordinate invariance failed: {label}"
        )
        invariance[label] = difference

if not checkpoints:
    checkpoint_path = RUN_ROOT / "epoch_00.pt"
    temporary = checkpoint_path.with_name(checkpoint_path.name + ".partial")
    torch.save({
        "epoch": 0,
        "config": CONFIG,
        "model_state_dict": MODEL.state_dict(),
        "optimizer_state_dict": OPTIMIZER.state_dict(),
        "history": [],
        "best": None,
    }, temporary)
    temporary.replace(checkpoint_path)
    print("Fresh ImageNet initialization saved: epoch_00.pt")

print("Run:", RUN_NAME)
print("Dataset sizes:", len(TRAIN_DATASET), len(VALID_DATASET))
print("Forward shape:", tuple(logits.shape))
print("Invalid-coordinate invariance:", invariance)
print("Next training epoch:", completed_epoch + 1)
print("Run directory:", RUN_ROOT)
print("M3 initialization, forward and geometry-mask checks: PASS")
print("No optimizer updates or test evaluation performed")

del smoke_loader, images, targets, points, valid, logits, altered_logits
torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


Fresh ImageNet initialization saved: epoch_00.pt
Run: m3_seed1
Dataset sizes: 162770 19867
Forward shape: (64, 24)
Invalid-coordinate invariance: {'large_finite': 0.0, 'nan': 0.0}
Next training epoch: 1
Run directory: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed1
M3 initialization, forward and geometry-mask checks: PASS
No optimizer updates or test evaluation performed


In [9]:
import json
import math
import random
import time

import numpy as np
import torch
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from evaluation.attributes import evaluate_attributes

required = (
    "MODEL", "OPTIMIZER", "CRITERION", "CONFIG",
    "TRAIN_DATASET", "VALID_DATASET", "RUN_ROOT", "DEVICE",
    "sha_file", "write_json_atomic",
)
missing = [name for name in required if name not in globals()]
assert not missing, f"Run the preparation and initialization cells first: {missing}"


def train_m3(
    model, optimizer, criterion, config,
    train_dataset, valid_dataset, run_root, device,
):
    saved_config = json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    )
    assert saved_config == config, "Run configuration mismatch"
    assert config["run_name"] == f"m3_seed{config['seed']}"
    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["selection_tie_rule"] == "earliest epoch"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )
    assert len(train_dataset) == 162770
    assert len(valid_dataset) == 19867
    assert train_dataset.failure_policy == "mask"
    assert valid_dataset.failure_policy == "mask"

    for relative, expected in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected, (
            f"Archived source differs: {relative}"
        )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def read_metric(metrics, name):
        matches = [
            value for key, value in metrics.items()
            if key.replace("_", "").lower() == name
        ]
        assert len(matches) == 1, (
            f"Unexpected metric keys: {list(metrics)}"
        )
        value = float(matches[0])
        assert math.isfinite(value), f"Non-finite metric: {name}"
        return value

    def select_best(history):
        if not history:
            return None
        # Python max preserves the earliest record on exact ties.
        row = max(history, key=lambda item: item["validation_mAP"])
        return {
            "epoch": row["epoch"],
            "checkpoint": f"epoch_{row['epoch']:02d}.pt",
            "validation_bce": row["validation_bce"],
            "validation_macro_f1": row["validation_macro_f1"],
            "validation_mAP": row["validation_mAP"],
            "f1_threshold": config["f1_threshold"],
        }

    probe_metrics = evaluate_attributes(
        np.array([[-10.0] * 24, [10.0] * 24]),
        np.array([[0] * 24, [1] * 24]),
        threshold=0.5,
    )
    assert read_metric(probe_metrics, "macrof1") == 1.0
    assert read_metric(probe_metrics, "map") == 1.0

    # Reload saved state on every invocation.
    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Initialization checkpoint is missing"
    latest = checkpoints[-1]

    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )

    assert checkpoint["config"] == config
    completed_epoch = int(checkpoint["epoch"])
    assert latest.name == f"epoch_{completed_epoch:02d}.pt"
    assert 0 <= completed_epoch <= config["epochs"]

    history = checkpoint["history"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    ), "Checkpoint history is not continuous"

    best = select_best(history)
    assert checkpoint["best"] == best, "Checkpoint selection mismatch"

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    del checkpoint
    torch.cuda.empty_cache()

    # Checkpoint history is authoritative after interrupted summary writes.
    write_json_atomic(run_root / "history.json", history)
    if best is not None:
        write_json_atomic(run_root / "best.json", best)

    # Preserve the historical M3 loader settings: no explicit generator
    # or custom worker initializer.
    valid_loader = DataLoader(
        valid_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=True,
        drop_last=False,
    )

    print("Run:", config["run_name"], flush=True)
    print("Restored:", latest.name, flush=True)
    print("Completed epochs:", completed_epoch, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        train_loader = DataLoader(
            train_dataset,
            batch_size=config["batch_size"],
            shuffle=True,
            num_workers=config["num_workers"],
            pin_memory=True,
            drop_last=False,
        )

        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.perf_counter()

        model.train()
        train_loss_sum = 0.0
        train_count = 0
        train_missing = 0

        progress = tqdm(
            train_loader,
            desc=f"{config['run_name']} epoch {epoch:02d} train",
        )
        for images, labels, points, names, valid in progress:
            count = images.shape[0]
            train_missing += int((~valid).sum().item())

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            points = points.to(device, non_blocking=True)
            valid = valid.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images, points, valid)
            loss = criterion(logits, labels)
            assert torch.isfinite(loss), (
                f"Non-finite training loss in epoch {epoch}"
            )

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * count
            train_count += count
            progress.set_postfix(
                BCE=f"{train_loss_sum / train_count:.4f}"
            )

        assert train_count == 162770
        assert train_missing == 249

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        valid_missing = 0
        logits_parts = []
        target_parts = []

        with torch.inference_mode():
            for images, labels, points, names, valid in tqdm(
                valid_loader,
                desc=f"{config['run_name']} epoch {epoch:02d} valid",
            ):
                count = images.shape[0]
                valid_missing += int((~valid).sum().item())
                target_parts.append(labels.numpy())

                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                points = points.to(device, non_blocking=True)
                valid = valid.to(device, non_blocking=True)

                logits = model(images, points, valid)
                loss = criterion(logits, labels)
                assert torch.isfinite(logits).all()
                assert torch.isfinite(loss)

                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu().numpy())

        assert valid_count == 19867
        assert valid_missing == 32

        metrics = evaluate_attributes(
            np.concatenate(logits_parts),
            np.concatenate(target_parts),
            threshold=config["f1_threshold"],
        )
        macro_f1 = read_metric(metrics, "macrof1")
        validation_map = read_metric(metrics, "map")

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_bce": train_loss_sum / train_count,
            "validation_bce": valid_loss_sum / valid_count,
            "validation_macro_f1": macro_f1,
            "validation_mAP": validation_map,
            "train_images": train_count,
            "validation_images": valid_count,
            "train_missing_geometry": train_missing,
            "validation_missing_geometry": valid_missing,
            "elapsed_seconds": time.perf_counter() - started,
            "peak_allocated_gpu_gib": (
                torch.cuda.max_memory_allocated() / 1024**3
            ),
        }
        history.append(record)
        best = select_best(history)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "config": config,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "history": history,
            "best": best,
        }, temporary)
        temporary.replace(checkpoint_path)

        write_json_atomic(run_root / "history.json", history)
        write_json_atomic(run_root / "best.json", best)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_bce']:.6f} | "
            f"valid BCE={record['validation_bce']:.6f} | "
            f"F1@0.5={macro_f1:.6f} | "
            f"mAP={validation_map:.6f} | "
            f"{record['elapsed_seconds'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )
        del train_loader, logits_parts, target_parts

    print("Selected checkpoint:", run_root / best["checkpoint"])
    print("M3 training: COMPLETE")
    print("Clean-validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best


M3_HISTORY, M3_BEST = train_m3(
    model=MODEL,
    optimizer=OPTIMIZER,
    criterion=CRITERION,
    config=CONFIG,
    train_dataset=TRAIN_DATASET,
    valid_dataset=VALID_DATASET,
    run_root=RUN_ROOT,
    device=DEVICE,
)

Run: m3_seed1
Restored: epoch_00.pt
Completed epochs: 0
Next epoch: 1


m3_seed1 epoch 01 train:   0%|          | 0/2544 [00:00<?, ?it/s]

m3_seed1 epoch 01 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 01 saved | train BCE=0.212986 | valid BCE=0.191935 | F1@0.5=0.740084 | mAP=0.812870 | 11.7 min | best epoch=1


m3_seed1 epoch 02 train:   0%|          | 0/2544 [00:00<?, ?it/s]

...

m3_seed1 epoch 09 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 09 saved | train BCE=0.053385 | valid BCE=0.329419 | F1@0.5=0.714686 | mAP=0.773796 | 12.4 min | best epoch=3


m3_seed1 epoch 10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

m3_seed1 epoch 10 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 10 saved | train BCE=0.041796 | valid BCE=0.358297 | F1@0.5=0.719384 | mAP=0.774033 | 13.6 min | best epoch=3
Selected checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed1/epoch_03.pt
M3 training: COMPLETE
Clean-validation prediction export and threshold fitting remain pending
No test evaluation performed


In [10]:
from pathlib import Path
import hashlib
import json

import numpy as np
import torch
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from models.resnet18_geometry import ResNet18GeometryAttributes
from evaluation.attributes import evaluate_attributes

# Change this parameter for Seed2.
SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_NAME = f"m3_seed{SEED}"
RUN_ROOT = PROJECT / "runs" / RUN_NAME

assert "VALID_DATASET" in globals(), (
    "Run the M3 data preparation cell first"
)
assert torch.cuda.is_available(), "A GPU runtime is required"


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


config = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
best = json.loads(
    (RUN_ROOT / "best.json").read_text(encoding="utf-8")
)
history = json.loads(
    (RUN_ROOT / "history.json").read_text(encoding="utf-8")
)

assert config["seed"] == SEED
assert config["run_name"] == RUN_NAME
assert [row["epoch"] for row in history] == list(
    range(1, config["epochs"] + 1)
)
selected = max(history, key=lambda row: row["validation_mAP"])
assert selected["epoch"] == best["epoch"]
assert best["checkpoint"] == f"epoch_{best['epoch']:02d}.pt"

for relative, expected in config["source_sha256"].items():
    assert sha_file(RUN_ROOT / "source" / relative) == expected, (
        f"Archived source differs: {relative}"
    )

# Verify the source files actually imported in this runtime.
import sys

for module_name, relative in (
    ("models.resnet18_geometry", "models/resnet18_geometry.py"),
    ("evaluation.attributes", "evaluation/attributes.py"),
    ("datasets.celeba_predicted", "datasets/celeba_predicted.py"),
    ("datasets.celeba", "datasets/celeba.py"),
    ("datasets.preprocessing", "datasets/preprocessing.py"),
    ("datasets.landmark_cache", "datasets/landmark_cache.py"),
):
    assert sha_file(sys.modules[module_name].__file__) == (
        config["source_sha256"][relative]
    ), f"Imported source differs: {module_name}"

assert len(VALID_DATASET) == 19867
assert VALID_DATASET.failure_policy == "mask"
assert VALID_DATASET.attribute_names == config["attribute_names"]

checkpoint_path = RUN_ROOT / best["checkpoint"]
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )
assert checkpoint["config"] == config
assert checkpoint["epoch"] == best["epoch"]
assert checkpoint["history"] == history[:best["epoch"]]

device = torch.device("cuda")
torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

export_model = ResNet18GeometryAttributes(pretrained=False).to(device)
export_model.load_state_dict(
    checkpoint["model_state_dict"], strict=True
)
del checkpoint
export_model.eval()

loader = DataLoader(
    VALID_DATASET,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"],
    pin_memory=True,
    drop_last=False,
)

logits_parts = []
target_parts = []
valid_parts = []
filenames = []

with torch.inference_mode():
    for images, targets, points, names, valid in tqdm(
        loader, desc=f"{RUN_NAME} selected validation"
    ):
        logits = export_model(
            images.to(device, non_blocking=True),
            points.to(device, non_blocking=True),
            valid.to(device, non_blocking=True),
        )
        assert logits.shape == (len(names), 24)
        assert torch.isfinite(logits).all()

        logits_parts.append(logits.cpu().numpy())
        target_parts.append(targets.numpy())
        valid_parts.append(valid.numpy())
        filenames.extend(names)

payload = {
    "filenames": np.asarray(filenames),
    "logits": np.concatenate(logits_parts),
    "targets": np.concatenate(target_parts),
    "geometry_valid": np.concatenate(valid_parts),
    "attribute_names": np.asarray(config["attribute_names"]),
}

assert filenames == VALID_DATASET.names
assert payload["logits"].shape == payload["targets"].shape == (19867, 24)
assert payload["geometry_valid"].dtype == np.bool_
assert int((~payload["geometry_valid"]).sum()) == 32

reference_path = (
    PROJECT / "runs/r18_seed0/day7/validation_predictions.npz"
)
with np.load(reference_path, allow_pickle=False) as reference:
    for key in ("filenames", "targets", "attribute_names"):
        assert np.array_equal(payload[key], reference[key]), (
            f"Historical validation alignment differs: {key}"
        )

metrics = evaluate_attributes(
    payload["logits"], payload["targets"], threshold=0.5
)
assert abs(metrics["macro_f1"] - best["validation_macro_f1"]) <= 1e-10
assert abs(metrics["map"] - best["validation_mAP"]) <= 1e-10

prediction_path = RUN_ROOT / "validation_predictions.npz"
if prediction_path.exists():
    with np.load(prediction_path, allow_pickle=False) as existing:
        assert set(existing.files) == set(payload)
        for key, value in payload.items():
            assert np.array_equal(existing[key], value), (
                f"Existing predictions differ; preserved: {key}"
            )
else:
    temporary = prediction_path.with_name(
        prediction_path.name + ".partial"
    )
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, **payload)
    temporary.replace(prediction_path)

audit = {
    "run_name": RUN_NAME,
    "checkpoint": best["checkpoint"],
    "selected_epoch": best["epoch"],
    "checkpoint_sha256": sha_file(checkpoint_path),
    "config_sha256": sha_file(RUN_ROOT / "config.json"),
    "prediction_sha256": sha_file(prediction_path),
    "samples": 19867,
    "masked_validation_samples": 32,
    "attribute_names": config["attribute_names"],
    "macro_f1_at_0.5": float(metrics["macro_f1"]),
    "map": float(metrics["map"]),
    "source_sha256": config["source_sha256"],
    "passed": True,
    "threshold_fitting_performed": False,
    "test_evaluated": False,
}
audit_path = RUN_ROOT / "validation_export_audit.json"
if audit_path.exists():
    assert json.loads(audit_path.read_text(encoding="utf-8")) == audit, (
        "Existing export audit differs; preserved"
    )
else:
    temporary = audit_path.with_name(audit_path.name + ".partial")
    temporary.write_text(
        json.dumps(audit, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(audit_path)

print("Full validation IDs, targets and attribute order: PASS")
print("Masked validation samples:", 32)
print("Selected-checkpoint metric reproduction: PASS")
print(f"F1@0.5: {metrics['macro_f1']:.6f}")
print(f"mAP: {metrics['map']:.6f}")
print("Saved:", prediction_path)
print("M3 selected-validation prediction export: PASS")
print("No training, threshold fitting or test evaluation performed")

del export_model, loader, logits_parts, target_parts, valid_parts
torch.cuda.empty_cache()

m3_seed1 selected validation:   0%|          | 0/311 [00:00<?, ?it/s]

Full validation IDs, targets and attribute order: PASS
Masked validation samples: 32
Selected-checkpoint metric reproduction: PASS
F1@0.5: 0.729975
mAP: 0.819263
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/m3_seed1/validation_predictions.npz
M3 selected-validation prediction export: PASS
No training, threshold fitting or test evaluation performed


In [11]:
from pathlib import Path
import hashlib
import json

import numpy as np
from sklearn.metrics import average_precision_score, f1_score

# Change this parameter for Seed2.
SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_NAME = f"m3_seed{SEED}"
RUN_ROOT = PROJECT / "runs" / RUN_NAME
PREDICTION_PATH = RUN_ROOT / "validation_predictions.npz"
OUTPUT_PATH = (
    PROJECT / "runs/benchmark_preparation"
    / f"seed{SEED}" / f"{RUN_NAME}.json"
)


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


config = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
best = json.loads(
    (RUN_ROOT / "best.json").read_text(encoding="utf-8")
)
audit = json.loads(
    (RUN_ROOT / "validation_export_audit.json").read_text(
        encoding="utf-8"
    )
)

assert config["seed"] == SEED
assert config["run_name"] == RUN_NAME
assert config["selection_metric"] == "validation_mAP"
assert config["selection_tie_rule"] == "earliest epoch"
assert audit["passed"] is True
assert audit["test_evaluated"] is False
assert audit["run_name"] == RUN_NAME
assert audit["checkpoint"] == best["checkpoint"]
assert audit["selected_epoch"] == best["epoch"]
assert best["checkpoint"] == f"epoch_{best['epoch']:02d}.pt"

checkpoint_path = RUN_ROOT / best["checkpoint"]
assert sha_file(RUN_ROOT / "config.json") == audit["config_sha256"]
assert sha_file(PREDICTION_PATH) == audit["prediction_sha256"]

for relative, expected in config["source_sha256"].items():
    assert sha_file(RUN_ROOT / "source" / relative) == expected, (
        f"Archived source differs: {relative}"
    )

with np.load(PREDICTION_PATH, allow_pickle=False) as saved:
    filenames = saved["filenames"].copy()
    logits = saved["logits"].copy()
    targets = saved["targets"].copy()
    geometry_valid = saved["geometry_valid"].copy()
    attribute_names = saved["attribute_names"].tolist()

assert logits.shape == targets.shape == (19867, 24)
assert filenames.shape == (19867,)
assert len(set(filenames.tolist())) == 19867
assert np.isfinite(logits).all()
assert np.isin(targets, [0, 1]).all()
assert attribute_names == config["attribute_names"]
assert geometry_valid.shape == (19867,)
assert geometry_valid.dtype == np.bool_
assert int((~geometry_valid).sum()) == 32
assert audit["masked_validation_samples"] == 32
assert np.all((targets.sum(axis=0) > 0) & (targets.sum(axis=0) < 19867))

reference_path = (
    PROJECT / "runs/r18_seed0/day7/validation_predictions.npz"
)
with np.load(reference_path, allow_pickle=False) as reference:
    assert np.array_equal(filenames, reference["filenames"])
    assert np.array_equal(targets, reference["targets"])
    assert attribute_names == reference["attribute_names"].tolist()

print("Historical validation IDs, targets and attribute order: PASS")

# Use the same stable float64 sigmoid and threshold grid as earlier runs.
values = logits.astype(np.float64)
probabilities = np.empty_like(values)
positive = values >= 0
probabilities[positive] = 1.0 / (1.0 + np.exp(-values[positive]))
exp_values = np.exp(values[~positive])
probabilities[~positive] = exp_values / (1.0 + exp_values)

grid_integer = list(range(5, 96))
grid = [value / 100 for value in grid_integer]
thresholds = []
baseline_f1 = []
tuned_f1 = []
attribute_ap = []

for column in range(24):
    labels = targets[:, column]
    scores = probabilities[:, column]

    baseline_f1.append(float(f1_score(
        labels, scores >= 0.5, zero_division=0
    )))
    attribute_ap.append(float(average_precision_score(labels, scores)))

    candidate_f1 = [
        float(f1_score(labels, scores >= threshold, zero_division=0))
        for threshold in grid
    ]
    selected = min(
        range(len(grid)),
        key=lambda index: (
            -candidate_f1[index],
            abs(grid_integer[index] - 50),
            grid_integer[index],
        ),
    )
    thresholds.append(grid[selected])
    tuned_f1.append(candidate_f1[selected])

baseline_macro_f1 = float(np.mean(baseline_f1))
tuned_macro_f1 = float(np.mean(tuned_f1))
mean_ap = float(np.mean(attribute_ap))

assert abs(baseline_macro_f1 - best["validation_macro_f1"]) <= 1e-10
assert abs(mean_ap - best["validation_mAP"]) <= 1e-10
assert all(
    tuned >= baseline
    for tuned, baseline in zip(tuned_f1, baseline_f1)
)

print("Selected-checkpoint metric reproduction: PASS")
print("Computing checkpoint SHA256...", flush=True)
checkpoint_hash = sha_file(checkpoint_path)
assert checkpoint_hash == audit["checkpoint_sha256"]

report = {
    "run_name": RUN_NAME,
    "checkpoint": best["checkpoint"],
    "selected_epoch": best["epoch"],
    "checkpoint_sha256": checkpoint_hash,
    "config_sha256": audit["config_sha256"],
    "prediction_path": str(PREDICTION_PATH),
    "prediction_sha256": audit["prediction_sha256"],
    "fit_split": "valid",
    "samples": 19867,
    "masked_validation_samples": 32,
    "geometry_pipeline": (
        "Cached MTCNN clean-image landmarks; shared preprocessing; "
        "retain all samples and mask invalid geometry after MLP."
    ),
    "attribute_names": attribute_names,
    "thresholds": thresholds,
    "objective": "per_attribute_F1",
    "grid": grid,
    "tie_break": "closest_to_0.5_then_lower",
    "comparison": ">=",
    "baseline_macro_f1": baseline_macro_f1,
    "tuned_macro_f1": tuned_macro_f1,
    "map": mean_ap,
    "per_attribute_f1_at_0.5": baseline_f1,
    "per_attribute_f1_tuned": tuned_f1,
    "per_attribute_ap": attribute_ap,
    "source_sha256": config["source_sha256"],
    "environment": config["environment"],
    "test_evaluated": False,
    "note": (
        "Thresholds fitted on clean validation only, including all "
        "32 masked-geometry samples. Tuned validation F1 is an "
        "in-sample fitting result, not independent evaluation. "
        "Reuse these thresholds unchanged for all clean and "
        "corrupted test conditions."
    ),
}

if OUTPUT_PATH.exists():
    assert json.loads(
        OUTPUT_PATH.read_text(encoding="utf-8")
    ) == report, f"Existing threshold report differs; preserved: {OUTPUT_PATH}"
else:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary = OUTPUT_PATH.with_name(OUTPUT_PATH.name + ".partial")
    temporary.write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(OUTPUT_PATH)

print("Run:", RUN_NAME)
print(f"F1@0.5: {baseline_macro_f1:.6f}")
print(f"F1(clean-validation tuned): {tuned_macro_f1:.6f}")
print(f"mAP: {mean_ap:.6f}")
print("Frozen thresholds:", len(thresholds))
print("Masked validation samples:", 32)
print("Saved:", OUTPUT_PATH)
print("M3 clean-validation threshold freeze: PASS")
print("No model inference, training or test evaluation performed")

Historical validation IDs, targets and attribute order: PASS
Selected-checkpoint metric reproduction: PASS
Computing checkpoint SHA256...
Run: m3_seed1
F1@0.5: 0.729975
F1(clean-validation tuned): 0.762818
mAP: 0.819263
Frozen thresholds: 24
Masked validation samples: 32
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/benchmark_preparation/seed1/m3_seed1.json
M3 clean-validation threshold freeze: PASS
No model inference, training or test evaluation performed


In [12]:
from pathlib import Path
import hashlib
import importlib
import json
import os
import shutil
import sys

import torch

# Change only this parameter for Seed2.
SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_NAME = f"oracle_seed{SEED}"
RUN_ROOT = PROJECT / "runs" / RUN_NAME
REFERENCE_ROOT = PROJECT / "runs/oracle_seed0"
WORKSPACE = Path("/content/multiseed_workspaces") / RUN_NAME
DATA_ROOT = Path("/content/seed1_shared_data/raw")
M3_ROOT = PROJECT / "runs" / f"m3_seed{SEED}"

assert torch.cuda.is_available(), "A GPU runtime is required"
assert REFERENCE_ROOT.is_dir(), "Historical Oracle run is missing"
assert DATA_ROOT.is_dir(), "Run the shared data preparation cell first"


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


reference = json.loads(
    (REFERENCE_ROOT / "config.json").read_text(encoding="utf-8")
)
m3_config = json.loads(
    (M3_ROOT / "config.json").read_text(encoding="utf-8")
)
preparation_path = M3_ROOT / "data_preparation.json"
assert sha_file(preparation_path) == m3_config["data_preparation_sha256"]
shared_preparation = json.loads(
    preparation_path.read_text(encoding="utf-8")
)
assert shared_preparation["seed"] == SEED
assert shared_preparation["train_samples"] == 162770
assert shared_preparation["validation_samples"] == 19867

assert reference["geometry_source"] == (
    "CelebA aligned GT; shared preprocessing"
)
assert reference["failure_policy"] == "GT available; retain every sample"
assert reference["geometry_valid"] == (
    "True for every train/validation sample"
)

for filename, expected in shared_preparation["annotation_sha256"].items():
    assert sha_file(DATA_ROOT / filename) == expected, (
        f"Shared annotation differs: {filename}"
    )

for relative, expected in reference["source_sha256"].items():
    source = REFERENCE_ROOT / "source" / relative
    destination = WORKSPACE / relative
    assert sha_file(source) == expected, (
        f"Historical Oracle source differs: {relative}"
    )
    if destination.exists():
        assert sha_file(destination) == expected, (
            f"Existing workspace source differs: {relative}"
        )
    else:
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source, destination)

for package in ("datasets", "models", "evaluation"):
    initializer = WORKSPACE / package / "__init__.py"
    if not initializer.exists():
        initializer.write_text("", encoding="utf-8")

for module_name in list(sys.modules):
    if any(
        module_name == package or module_name.startswith(package + ".")
        for package in ("datasets", "models", "evaluation")
    ):
        del sys.modules[module_name]

workspace_string = str(WORKSPACE)
if workspace_string in sys.path:
    sys.path.remove(workspace_string)
sys.path.insert(0, workspace_string)
os.chdir(WORKSPACE)
importlib.invalidate_caches()

from datasets.celeba import CelebAAttributes

TRAIN_DATASET = CelebAAttributes(split="train", root=DATA_ROOT)
VALID_DATASET = CelebAAttributes(split="valid", root=DATA_ROOT)

rows = [
    line.split()
    for line in (DATA_ROOT / "list_eval_partition.txt").read_text(
        encoding="utf-8-sig"
    ).splitlines()
]
assert TRAIN_DATASET.names == [
    name for name, split in rows if split == "0"
]
assert VALID_DATASET.names == [
    name for name, split in rows if split == "1"
]
assert len(TRAIN_DATASET) == 162770
assert len(VALID_DATASET) == 19867
assert TRAIN_DATASET.attribute_names == reference["attribute_names"]
assert VALID_DATASET.attribute_names == reference["attribute_names"]

image_root = DATA_ROOT / "img_align_celeba"
for name in TRAIN_DATASET.names + VALID_DATASET.names:
    assert (image_root / name).is_file(), f"Missing shared image: {name}"

# GT remains available for a historical MTCNN detection failure.
failure_index = TRAIN_DATASET.names.index("000199.jpg")
image, targets, gt_points, filename = TRAIN_DATASET[failure_index]
assert filename == "000199.jpg"
assert image.shape == (3, 224, 224)
assert targets.shape == (24,)
assert gt_points.shape == (10,)
assert torch.isfinite(image).all()
assert torch.isfinite(gt_points).all()

ORACLE_PREPARATION = {
    "run_name": RUN_NAME,
    "seed": SEED,
    "reference_config_sha256": sha_file(REFERENCE_ROOT / "config.json"),
    "source_sha256": reference["source_sha256"],
    "shared_preparation_sha256": sha_file(preparation_path),
    "image_zip_sha256": shared_preparation["image_zip_sha256"],
    "annotation_sha256": shared_preparation["annotation_sha256"],
    "train_samples": 162770,
    "validation_samples": 19867,
    "geometry_source": reference["geometry_source"],
    "masked_train_samples": 0,
    "masked_validation_samples": 0,
}

print("Historical Oracle source snapshot: PASS")
print("Shared annotation hashes and official dataset order: PASS")
print("Previously verified local training/validation images: REUSED")
print("Historical MTCNN failure sample retained with finite GT: PASS")
print("Run:", RUN_NAME)
print("Dataset sizes:", len(TRAIN_DATASET), len(VALID_DATASET))
print("Geometry: GT; every sample valid")
print("Workspace:", WORKSPACE)
print("Existing run directory:", RUN_ROOT.exists())
print("Oracle shared data and source preparation: PASS")
print("No model initialization, optimizer updates or test evaluation performed")

Historical Oracle source snapshot: PASS
Shared annotation hashes and official dataset order: PASS
Previously verified local training/validation images: REUSED
Historical MTCNN failure sample retained with finite GT: PASS
Run: oracle_seed1
Dataset sizes: 162770 19867
Geometry: GT; every sample valid
Workspace: /content/multiseed_workspaces/oracle_seed1
Existing run directory: False
Oracle shared data and source preparation: PASS
No model initialization, optimizer updates or test evaluation performed


In [13]:
import copy
import gc
import hashlib
import importlib.metadata
import json
import platform
import random
import shutil

import numpy as np
import torch
from torch import nn
from torch.torch_version import TorchVersion

from models.resnet18_geometry import ResNet18GeometryAttributes

required = (
    "SEED", "PROJECT", "RUN_NAME", "RUN_ROOT", "REFERENCE_ROOT",
    "WORKSPACE", "TRAIN_DATASET", "VALID_DATASET",
    "ORACLE_PREPARATION", "sha_file",
)
missing = [name for name in required if name not in globals()]
assert not missing, f"Run the Oracle preparation cell first: {missing}"
assert RUN_NAME == f"oracle_seed{SEED}"


def write_json_atomic(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


# Release previous training objects before creating the Oracle model.
for name in (
    "MODEL", "OPTIMIZER", "CRITERION", "RGB_CONTEXT",
    "model", "optimizer", "export_model",
):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": importlib.metadata.version("torchvision"),
    "numpy": importlib.metadata.version("numpy"),
    "Pillow": importlib.metadata.version("Pillow"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}
runtime = {
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
}

m3_config = json.loads(
    (PROJECT / "runs" / f"m3_seed{SEED}" / "config.json").read_text(
        encoding="utf-8"
    )
)
assert environment == m3_config["environment"], (
    "Environment differs from the corresponding M3 run"
)
assert runtime == m3_config["runtime"], (
    "CUDA, cuDNN or GPU differs from the corresponding M3 run"
)

reference = json.loads(
    (REFERENCE_ROOT / "config.json").read_text(encoding="utf-8")
)
CONFIG = copy.deepcopy(reference)
CONFIG.update({
    "run_name": RUN_NAME,
    "seed": SEED,
    "environment": environment,
    "runtime": runtime,
    "reference_run": "oracle_seed0",
    "reference_config_sha256": ORACLE_PREPARATION[
        "reference_config_sha256"
    ],
    "reference_environment": reference["environment"],
    "data_preparation_sha256": hashlib.sha256(
        (
            json.dumps(ORACLE_PREPARATION, indent=2, allow_nan=False)
            + "\n"
        ).encode("utf-8")
    ).hexdigest(),
})

RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Check every existing destination before archiving records.
for filename, value in (
    ("config.json", CONFIG),
    ("data_preparation.json", ORACLE_PREPARATION),
):
    path = RUN_ROOT / filename
    if path.exists():
        assert json.loads(path.read_text(encoding="utf-8")) == value, (
            f"Existing record differs; preserved: {path}"
        )

for relative, expected in CONFIG["source_sha256"].items():
    source = WORKSPACE / relative
    destination = RUN_ROOT / "source" / relative
    assert sha_file(source) == expected
    if destination.exists():
        assert sha_file(destination) == expected, (
            f"Archived source differs: {relative}"
        )

for relative, expected in CONFIG["source_sha256"].items():
    destination = RUN_ROOT / "source" / relative
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(WORKSPACE / relative, destination)

for filename, value in (
    ("config.json", CONFIG),
    ("data_preparation.json", ORACLE_PREPARATION),
):
    path = RUN_ROOT / filename
    if not path.exists():
        write_json_atomic(path, value)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = CONFIG["cudnn_deterministic"]
torch.backends.cudnn.benchmark = CONFIG["cudnn_benchmark"]

DEVICE = torch.device("cuda")
checkpoints = sorted(RUN_ROOT.glob("epoch_[0-9][0-9].pt"))

MODEL = ResNet18GeometryAttributes(
    num_attributes=24,
    pretrained=not bool(checkpoints),
).to(DEVICE)
OPTIMIZER = torch.optim.Adam(
    MODEL.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
CRITERION = nn.BCEWithLogitsLoss()

assert all(parameter.requires_grad for parameter in MODEL.parameters())
assert sum(parameter.numel() for parameter in MODEL.parameters()) == (
    11_191_000
)

if checkpoints:
    latest = checkpoints[-1]
    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )
    assert checkpoint["config"] == CONFIG
    completed_epoch = checkpoint["epoch"]
    assert latest.name == f"epoch_{completed_epoch:02d}.pt"
    assert 0 <= completed_epoch <= CONFIG["epochs"]
    assert [row["epoch"] for row in checkpoint["history"]] == list(
        range(1, completed_epoch + 1)
    )
    MODEL.load_state_dict(checkpoint["model_state_dict"], strict=True)
    OPTIMIZER.load_state_dict(checkpoint["optimizer_state_dict"])
    del checkpoint
    print("Restored:", latest.name)
else:
    completed_epoch = 0
    checkpoint_path = RUN_ROOT / "epoch_00.pt"
    temporary = checkpoint_path.with_name(
        checkpoint_path.name + ".partial"
    )
    torch.save({
        "epoch": 0,
        "config": CONFIG,
        "history": [],
        "best": {"epoch": None, "map": None},
        "model_state_dict": MODEL.state_dict(),
        "optimizer_state_dict": OPTIMIZER.state_dict(),
    }, temporary)
    temporary.replace(checkpoint_path)
    print("Fresh ImageNet initialization saved: epoch_00.pt")

# The Oracle uses GT geometry for every sample, including detector failures.
failure_index = TRAIN_DATASET.names.index("000199.jpg")
samples = [
    TRAIN_DATASET[index] for index in (0, 1, 2, failure_index)
]
images = torch.stack([sample[0] for sample in samples]).to(DEVICE)
points = torch.stack([sample[2] for sample in samples]).to(DEVICE)
valid = torch.ones(4, dtype=torch.bool, device=DEVICE)

MODEL.eval()
with torch.inference_mode():
    logits = MODEL(images, points, valid)
assert logits.shape == (4, 24)
assert torch.isfinite(logits).all()
assert torch.isfinite(points).all()

print("Run:", RUN_NAME)
print("Dataset sizes:", len(TRAIN_DATASET), len(VALID_DATASET))
print("Geometry: GT; all samples valid")
print("Forward shape:", tuple(logits.shape))
print("Next training epoch:", completed_epoch + 1)
print("Run directory:", RUN_ROOT)
print("Oracle initialization and GT forward check: PASS")
print("No optimizer updates or test evaluation performed")

del images, points, valid, logits, samples
torch.cuda.empty_cache()

Fresh ImageNet initialization saved: epoch_00.pt
Run: oracle_seed1
Dataset sizes: 162770 19867
Geometry: GT; all samples valid
Forward shape: (4, 24)
Next training epoch: 1
Run directory: /content/drive/MyDrive/FacialVisualProfile/runs/oracle_seed1
Oracle initialization and GT forward check: PASS
No optimizer updates or test evaluation performed


In [ ]:
import json
import random
import time

import numpy as np
import torch
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from evaluation.attributes import evaluate_attributes

required = (
    "MODEL", "OPTIMIZER", "CRITERION", "CONFIG",
    "TRAIN_DATASET", "VALID_DATASET", "RUN_ROOT", "DEVICE",
    "sha_file", "write_json_atomic",
)
missing = [name for name in required if name not in globals()]
assert not missing, (
    f"Run the Oracle preparation and initialization cells first: {missing}"
)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def train_oracle(
    model, optimizer, criterion, config,
    train_dataset, valid_dataset, run_root, device,
):
    assert config["run_name"] == f"oracle_seed{config['seed']}"
    assert json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    ) == config
    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["selection_tie_rule"] == "earliest epoch"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )
    assert config["geometry_source"] == (
        "CelebA aligned GT; shared preprocessing"
    )
    assert type(train_dataset).__name__ == "CelebAAttributes"
    assert type(valid_dataset).__name__ == "CelebAAttributes"
    assert len(train_dataset) == 162770
    assert len(valid_dataset) == 19867
    assert train_dataset.attribute_names == config["attribute_names"]
    assert valid_dataset.attribute_names == config["attribute_names"]

    for relative, expected in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected, (
            f"Archived source differs: {relative}"
        )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def select_best(history):
        if not history:
            return {"epoch": None, "map": None}
        row = max(history, key=lambda item: item["validation_map"])
        return {"epoch": row["epoch"], "map": row["validation_map"]}

    def make_loader(dataset, shuffle):
        # Preserve the historical Oracle loader settings.
        return DataLoader(
            dataset,
            batch_size=config["batch_size"],
            shuffle=shuffle,
            num_workers=config["num_workers"],
            pin_memory=True,
            drop_last=False,
            worker_init_fn=seed_worker,
        )

    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Initialization checkpoint is missing"
    latest = checkpoints[-1]

    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )
    assert checkpoint["config"] == config

    completed_epoch = checkpoint["epoch"]
    history = checkpoint["history"]
    assert latest.name == f"epoch_{completed_epoch:02d}.pt"
    assert 0 <= completed_epoch <= config["epochs"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    )

    best = select_best(history)
    assert checkpoint["best"] == best, "Checkpoint selection mismatch"

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    del checkpoint
    torch.cuda.empty_cache()

    write_json_atomic(run_root / "history.json", history)
    if history:
        write_json_atomic(run_root / "best.json", best)

    print("Run:", config["run_name"], flush=True)
    print("Restored:", latest.name, flush=True)
    print("Completed epochs:", completed_epoch, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)
    print("Geometry: GT; all samples valid", flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        torch.cuda.synchronize()
        started = time.perf_counter()
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for images, targets, points, names in tqdm(
            make_loader(train_dataset, True),
            desc=f"{config['run_name']} epoch {epoch:02d} train",
        ):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            points = points.to(device, non_blocking=True)
            valid = torch.ones(
                images.shape[0], dtype=torch.bool, device=device
            )

            optimizer.zero_grad(set_to_none=True)
            logits = model(images, points, valid)
            loss = criterion(logits, targets)
            assert torch.isfinite(loss), "Non-finite training loss"

            loss.backward()
            optimizer.step()

            count = images.shape[0]
            train_loss_sum += loss.item() * count
            train_count += count

        assert train_count == 162770

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        logits_parts = []
        target_parts = []
        validation_names = []

        with torch.inference_mode():
            for images, targets, points, names in tqdm(
                make_loader(valid_dataset, False),
                desc=f"{config['run_name']} epoch {epoch:02d} valid",
            ):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                points = points.to(device, non_blocking=True)
                valid = torch.ones(
                    images.shape[0], dtype=torch.bool, device=device
                )

                logits = model(images, points, valid)
                loss = criterion(logits, targets)
                assert torch.isfinite(logits).all()
                assert torch.isfinite(loss)

                count = images.shape[0]
                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu().numpy())
                target_parts.append(targets.cpu().numpy())
                validation_names.extend(names)

        assert valid_count == 19867
        assert validation_names == valid_dataset.names

        metrics = evaluate_attributes(
            np.concatenate(logits_parts),
            np.concatenate(target_parts),
            threshold=config["f1_threshold"],
        )
        assert np.isfinite(metrics["macro_f1"])
        assert np.isfinite(metrics["map"])

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_bce": train_loss_sum / train_count,
            "validation_bce": valid_loss_sum / valid_count,
            "validation_macro_f1_at_0.5": float(metrics["macro_f1"]),
            "validation_map": float(metrics["map"]),
            "seconds": time.perf_counter() - started,
        }
        history.append(record)
        best = select_best(history)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "config": config,
            "history": history,
            "best": best,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }, temporary)
        temporary.replace(checkpoint_path)

        write_json_atomic(run_root / "history.json", history)
        write_json_atomic(run_root / "best.json", best)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_bce']:.6f} | "
            f"valid BCE={record['validation_bce']:.6f} | "
            f"F1@0.5={metrics['macro_f1']:.6f} | "
            f"mAP={metrics['map']:.6f} | "
            f"{record['seconds'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )
        del logits_parts, target_parts

    print(
        "Selected checkpoint:",
        run_root / f"epoch_{best['epoch']:02d}.pt",
    )
    print("Oracle training: COMPLETE")
    print("Validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best


ORACLE_HISTORY, ORACLE_BEST = train_oracle(
    model=MODEL,
    optimizer=OPTIMIZER,
    criterion=CRITERION,
    config=CONFIG,
    train_dataset=TRAIN_DATASET,
    valid_dataset=VALID_DATASET,
    run_root=RUN_ROOT,
    device=DEVICE,
)

Run: oracle_seed1
Restored: epoch_00.pt
Completed epochs: 0
Next epoch: 1
Geometry: GT; all samples valid


oracle_seed1 epoch 01 train:   0%|          | 0/2544 [00:01<?, ?it/s]

oracle_seed1 epoch 01 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 01 saved | train BCE=0.213012 | valid BCE=0.191638 | F1@0.5=0.739722 | mAP=0.812260 | 12.6 min | best epoch=1


oracle_seed1 epoch 02 train:   0%|          | 0/2544 [00:00<?, ?it/s]

...

Epoch 08 saved | train BCE=0.070219 | valid BCE=0.297753 | F1@0.5=0.719735 | mAP=0.778500 | 12.5 min | best epoch=3


oracle_seed1 epoch 09 train:   0%|          | 0/2544 [00:00<?, ?it/s]

oracle_seed1 epoch 09 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Epoch 09 saved | train BCE=0.053046 | valid BCE=0.327756 | F1@0.5=0.719184 | mAP=0.775698 | 13.6 min | best epoch=3


oracle_seed1 epoch 10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

In [1]:
from google.colab import drive
from pathlib import Path
import json
import math

import torch
from torch.torch_version import TorchVersion

drive.mount("/content/drive")

SEED = 1
RUN_ROOT = (
    Path("/content/drive/MyDrive/FacialVisualProfile")
    / "runs" / f"oracle_seed{SEED}"
)

config = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
assert config["run_name"] == f"oracle_seed{SEED}"
assert config["seed"] == SEED

checkpoints = sorted(RUN_ROOT.glob("epoch_[0-9][0-9].pt"))
assert checkpoints, "No saved checkpoints found"
print("Saved checkpoints:", [path.name for path in checkpoints])

latest = checkpoints[-1]
with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        latest, map_location="cpu", weights_only=True
    )

assert checkpoint["config"] == config
completed = checkpoint["epoch"]
history = checkpoint["history"]

assert latest.name == f"epoch_{completed:02d}.pt"
assert 0 <= completed <= config["epochs"]
assert [row["epoch"] for row in history] == list(
    range(1, completed + 1)
)

print("\nEpoch   Train BCE   Valid BCE   F1@0.5     mAP")
for row in history:
    values = (
        row["train_bce"],
        row["validation_bce"],
        row["validation_macro_f1_at_0.5"],
        row["validation_map"],
    )
    assert all(math.isfinite(value) for value in values)
    print(
        f"{row['epoch']:>5}   "
        f"{values[0]:.6f}    {values[1]:.6f}    "
        f"{values[2]:.6f}   {values[3]:.6f}"
    )

assert history, "No completed training epoch found"
selected = max(history, key=lambda row: row["validation_map"])
best = {
    "epoch": selected["epoch"],
    "map": selected["validation_map"],
}
assert checkpoint["best"] == best
del checkpoint

for filename, expected in (
    ("history.json", history),
    ("best.json", best),
):
    path = RUN_ROOT / filename
    if path.exists():
        actual = json.loads(path.read_text(encoding="utf-8"))
        print(
            f"{filename}:",
            "MATCH" if actual == expected else "DIFFERS",
        )
    else:
        print(f"{filename}: MISSING")

selected_path = RUN_ROOT / f"epoch_{best['epoch']:02d}.pt"
with torch.serialization.safe_globals([TorchVersion]):
    selected_checkpoint = torch.load(
        selected_path, map_location="cpu", weights_only=True
    )
assert selected_checkpoint["config"] == config
assert selected_checkpoint["epoch"] == best["epoch"]
assert selected_checkpoint["history"] == history[:best["epoch"]]
del selected_checkpoint

print("\nLatest verified checkpoint:", latest.name)
print("Completed epochs:", completed, "/", config["epochs"])
print("Selected checkpoint:", selected_path.name)
print(f"Selected F1@0.5: {selected['validation_macro_f1_at_0.5']:.6f}")
print(f"Selected mAP: {best['map']:.6f}")

if completed == config["epochs"]:
    assert all(
        (RUN_ROOT / f"epoch_{epoch:02d}.pt").is_file()
        for epoch in range(config["epochs"] + 1)
    )
    print("Oracle training completion: PASS")
    print("No further training is needed")
else:
    print("Oracle training is incomplete")
    print("Resume from epoch:", completed + 1)

print("Read-only inspection completed; no files or weights modified")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved checkpoints: ['epoch_00.pt', 'epoch_01.pt', 'epoch_02.pt', 'epoch_03.pt', 'epoch_04.pt', 'epoch_05.pt', 'epoch_06.pt', 'epoch_07.pt', 'epoch_08.pt', 'epoch_09.pt']

Epoch   Train BCE   Valid BCE   F1@0.5     mAP
    1   0.213012    0.191638    0.739722   0.812260
    2   0.186568    0.190788    0.735021   0.815474
    3   0.174713    0.192195    0.735095   0.819262
    4   0.160435    0.195598    0.744018   0.814094
    5   0.141726    0.212095    0.731081   0.804483
    6   0.118296    0.226323    0.728400   0.796068
    7   0.092983    0.260989    0.731560   0.787928
    8   0.070219    0.297753    0.719735   0.778500
    9   0.053046    0.327756    0.719184   0.775698
history.json: MATCH
best.json: MATCH

Latest verified checkpoint: epoch_09.pt
Completed epochs: 9 / 10
Selected checkpoint: epoch_03.pt
Selected F1@0.5: 0.735095
Selected mAP: 0.819262


In [3]:
from google.colab import drive
from pathlib import Path
import hashlib
import importlib
import importlib.metadata
import json
import os
import platform
import shutil
import sys
import zipfile

import numpy as np
import torch
from torch import nn
from torch.torch_version import TorchVersion

drive.mount("/content/drive")

SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_ROOT = PROJECT / "runs" / f"oracle_seed{SEED}"
WORKSPACE = Path("/content/multiseed_workspaces") / f"oracle_seed{SEED}"
DATA_ROOT = Path("/content/seed1_shared_data/raw")

assert torch.cuda.is_available(), "Use a GPU runtime"


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def copy_verified(source, destination, expected):
    source, destination = Path(source), Path(destination)
    if destination.exists():
        assert sha_file(destination) == expected, (
            f"Existing file differs; preserved: {destination}"
        )
    else:
        assert sha_file(source) == expected, (
            f"Source hash mismatch: {source}"
        )
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_name(destination.name + ".partial")
        shutil.copyfile(source, temporary)
        assert sha_file(temporary) == expected
        temporary.replace(destination)


CONFIG = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
assert CONFIG["run_name"] == f"oracle_seed{SEED}"
assert CONFIG["seed"] == SEED

environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": importlib.metadata.version("torchvision"),
    "numpy": importlib.metadata.version("numpy"),
    "Pillow": importlib.metadata.version("Pillow"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}
runtime = {
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
}
assert environment == CONFIG["environment"], (
    f"Environment mismatch: expected {CONFIG['environment']}; "
    f"current {environment}"
)
assert runtime == CONFIG["runtime"], "CUDA, cuDNN or GPU mismatch"

preparation_path = RUN_ROOT / "data_preparation.json"
assert sha_file(preparation_path) == CONFIG["data_preparation_sha256"]
preparation = json.loads(
    preparation_path.read_text(encoding="utf-8")
)
assert preparation["run_name"] == CONFIG["run_name"]
assert preparation["seed"] == SEED

checkpoints = sorted(RUN_ROOT.glob("epoch_[0-9][0-9].pt"))
assert checkpoints, "Saved checkpoint is missing"
latest = checkpoints[-1]

with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        latest, map_location="cpu", weights_only=True
    )
assert checkpoint["config"] == CONFIG
completed_epoch = checkpoint["epoch"]
assert latest.name == f"epoch_{completed_epoch:02d}.pt"
assert [row["epoch"] for row in checkpoint["history"]] == list(
    range(1, completed_epoch + 1)
)
assert 0 <= completed_epoch <= CONFIG["epochs"]

print("Checkpoint verified:", latest.name, flush=True)
print("Next epoch:", completed_epoch + 1, flush=True)

# Restore annotations bound to this Oracle run.
for filename, expected in preparation["annotation_sha256"].items():
    copy_verified(
        PROJECT / "data" / filename,
        DATA_ROOT / filename,
        expected,
    )

rows = [
    line.split()
    for line in (DATA_ROOT / "list_eval_partition.txt").read_text(
        encoding="utf-8-sig"
    ).splitlines()
]
assert all(
    len(row) == 2 and row[1] in ("0", "1", "2") for row in rows
)
assert len(rows) == len({name for name, split in rows}) == 202599
train_names = [name for name, split in rows if split == "0"]
valid_names = [name for name, split in rows if split == "1"]
assert len(train_names) == 162770
assert len(valid_names) == 19867
required_names = train_names + valid_names

candidates = (
    Path("/content/seed1_shared_data/img_align_celeba.zip"),
    Path("/content/r50_validation_data/img_align_celeba.zip"),
)
archive_path = next(
    (path for path in candidates if path.is_file()), candidates[0]
)
print("Restoring shared image archive...", flush=True)
copy_verified(
    PROJECT / "data/img_align_celeba.zip",
    archive_path,
    preparation["image_zip_sha256"],
)

image_root = DATA_ROOT / "img_align_celeba"
image_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive_path) as archive:
    members = {}
    for member in archive.infolist():
        if member.is_dir():
            continue
        name = Path(member.filename).name
        if name.lower().endswith(".jpg"):
            assert name not in members, f"Duplicate image ID: {name}"
            members[name] = member
    assert set(required_names).issubset(members)

    for index, name in enumerate(required_names, start=1):
        content = archive.read(members[name])
        destination = image_root / name
        if destination.exists():
            assert destination.read_bytes() == content, (
                f"Existing image differs; preserved: {name}"
            )
        else:
            temporary = destination.with_name(name + ".partial")
            temporary.write_bytes(content)
            temporary.replace(destination)

        if index % 20000 == 0 or index == len(required_names):
            print(
                f"Restored and verified images: "
                f"{index}/{len(required_names)}",
                flush=True,
            )

# Restore the source archived by this Oracle run.
for relative, expected in CONFIG["source_sha256"].items():
    copy_verified(
        RUN_ROOT / "source" / relative,
        WORKSPACE / relative,
        expected,
    )

for package in ("datasets", "models", "evaluation"):
    initializer = WORKSPACE / package / "__init__.py"
    if not initializer.exists():
        initializer.write_text("", encoding="utf-8")

for name in list(sys.modules):
    if any(
        name == package or name.startswith(package + ".")
        for package in ("datasets", "models", "evaluation")
    ):
        del sys.modules[name]

workspace_string = str(WORKSPACE)
if workspace_string in sys.path:
    sys.path.remove(workspace_string)
sys.path.insert(0, workspace_string)
os.chdir(WORKSPACE)
importlib.invalidate_caches()

from datasets.celeba import CelebAAttributes
from models.resnet18_geometry import ResNet18GeometryAttributes
from evaluation.attributes import evaluate_attributes

for module_name, relative in (
    ("datasets.celeba", "datasets/celeba.py"),
    ("datasets.preprocessing", "datasets/preprocessing.py"),
    ("models.resnet18_geometry", "models/resnet18_geometry.py"),
    ("evaluation.attributes", "evaluation/attributes.py"),
):
    assert sha_file(sys.modules[module_name].__file__) == (
        CONFIG["source_sha256"][relative]
    ), f"Imported source differs: {module_name}"

TRAIN_DATASET = CelebAAttributes(split="train", root=DATA_ROOT)
VALID_DATASET = CelebAAttributes(split="valid", root=DATA_ROOT)
assert TRAIN_DATASET.names == train_names
assert VALID_DATASET.names == valid_names
assert TRAIN_DATASET.attribute_names == CONFIG["attribute_names"]
assert VALID_DATASET.attribute_names == CONFIG["attribute_names"]

DEVICE = torch.device("cuda")
torch.backends.cudnn.deterministic = CONFIG["cudnn_deterministic"]
torch.backends.cudnn.benchmark = CONFIG["cudnn_benchmark"]

# Restore this run's trained weights without ImageNet initialization.
MODEL = ResNet18GeometryAttributes(pretrained=False).to(DEVICE)
MODEL.load_state_dict(checkpoint["model_state_dict"], strict=True)

OPTIMIZER = torch.optim.Adam(
    MODEL.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
OPTIMIZER.load_state_dict(checkpoint["optimizer_state_dict"])
if completed_epoch > 0:
    assert len(OPTIMIZER.state) > 0, "Adam state was not restored"

CRITERION = nn.BCEWithLogitsLoss()
del checkpoint

print("Environment and original data binding: PASS")
print("Training/validation data and archived source: PASS")
print("Strict model loading and Adam restoration: PASS")
print("Completed epochs:", completed_epoch)
print("Next epoch:", completed_epoch + 1)
print("Oracle runtime restoration: PASS")
print("No optimizer updates or test evaluation performed")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint verified: epoch_09.pt
Next epoch: 10
Restoring shared image archive...
Restored and verified images: 20000/182637
Restored and verified images: 40000/182637
Restored and verified images: 60000/182637
Restored and verified images: 80000/182637
Restored and verified images: 100000/182637
Restored and verified images: 120000/182637
Restored and verified images: 140000/182637
Restored and verified images: 160000/182637
Restored and verified images: 180000/182637
Restored and verified images: 182637/182637
Environment and original data binding: PASS
Training/validation data and archived source: PASS
Strict model loading and Adam restoration: PASS
Completed epochs: 9
Next epoch: 10
Oracle runtime restoration: PASS
No optimizer updates or test evaluation performed


In [4]:
import json
import random
import time

import numpy as np
import torch
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from evaluation.attributes import evaluate_attributes

required = (
    "MODEL", "OPTIMIZER", "CRITERION", "CONFIG",
    "TRAIN_DATASET", "VALID_DATASET", "RUN_ROOT", "DEVICE",
    "sha_file", "write_json_atomic",
)
missing = [name for name in required if name not in globals()]
assert not missing, f"Run the Oracle restoration cell first: {missing}"


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def train_oracle(
    model, optimizer, criterion, config,
    train_dataset, valid_dataset, run_root, device,
):
    assert config["run_name"] == f"oracle_seed{config['seed']}"
    assert json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    ) == config
    assert config["optimizer"] == "Adam"
    assert config["precision"] == "FP32"
    assert config["f1_threshold"] == 0.5
    assert config["selection_metric"] == "validation_mAP"
    assert config["selection_tie_rule"] == "earliest epoch"
    assert config["epoch_seed_rule"] == (
        "seed + epoch; epochs start at 1"
    )
    assert config["geometry_source"] == (
        "CelebA aligned GT; shared preprocessing"
    )
    assert type(train_dataset).__name__ == "CelebAAttributes"
    assert type(valid_dataset).__name__ == "CelebAAttributes"
    assert len(train_dataset) == 162770
    assert len(valid_dataset) == 19867
    assert train_dataset.attribute_names == config["attribute_names"]
    assert valid_dataset.attribute_names == config["attribute_names"]

    for relative, expected in config["source_sha256"].items():
        assert sha_file(run_root / "source" / relative) == expected, (
            f"Archived source differs: {relative}"
        )

    torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
    torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

    def select_best(history):
        if not history:
            return {"epoch": None, "map": None}
        row = max(history, key=lambda item: item["validation_map"])
        return {"epoch": row["epoch"], "map": row["validation_map"]}

    def make_loader(dataset, shuffle):
        # Preserve the historical Oracle loader settings.
        return DataLoader(
            dataset,
            batch_size=config["batch_size"],
            shuffle=shuffle,
            num_workers=config["num_workers"],
            pin_memory=True,
            drop_last=False,
            worker_init_fn=seed_worker,
        )

    checkpoints = sorted(run_root.glob("epoch_[0-9][0-9].pt"))
    assert checkpoints, "Saved checkpoint is missing"
    latest = checkpoints[-1]

    with torch.serialization.safe_globals([TorchVersion]):
        checkpoint = torch.load(
            latest, map_location="cpu", weights_only=True
        )
    assert checkpoint["config"] == config

    completed_epoch = checkpoint["epoch"]
    history = checkpoint["history"]
    assert latest.name == f"epoch_{completed_epoch:02d}.pt"
    assert 0 <= completed_epoch <= config["epochs"]
    assert [row["epoch"] for row in history] == list(
        range(1, completed_epoch + 1)
    )

    best = select_best(history)
    assert checkpoint["best"] == best, "Checkpoint selection mismatch"

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    del checkpoint
    torch.cuda.empty_cache()

    write_json_atomic(run_root / "history.json", history)
    if history:
        write_json_atomic(run_root / "best.json", best)

    print("Run:", config["run_name"], flush=True)
    print("Restored:", latest.name, flush=True)
    print("Completed epochs:", completed_epoch, flush=True)
    print("Next epoch:", completed_epoch + 1, flush=True)
    print("Geometry: GT; all samples valid", flush=True)

    for epoch in range(completed_epoch + 1, config["epochs"] + 1):
        epoch_seed = config["seed"] + epoch
        random.seed(epoch_seed)
        np.random.seed(epoch_seed)
        torch.manual_seed(epoch_seed)
        torch.cuda.manual_seed_all(epoch_seed)

        torch.cuda.synchronize()
        started = time.perf_counter()
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for images, targets, points, names in tqdm(
            make_loader(train_dataset, True),
            desc=f"{config['run_name']} epoch {epoch:02d} train",
        ):
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            points = points.to(device, non_blocking=True)
            valid = torch.ones(
                images.shape[0], dtype=torch.bool, device=device
            )

            optimizer.zero_grad(set_to_none=True)
            logits = model(images, points, valid)
            loss = criterion(logits, targets)
            assert torch.isfinite(loss), "Non-finite training loss"

            loss.backward()
            optimizer.step()

            count = images.shape[0]
            train_loss_sum += loss.item() * count
            train_count += count

        assert train_count == 162770

        model.eval()
        valid_loss_sum = 0.0
        valid_count = 0
        logits_parts = []
        target_parts = []
        validation_names = []

        with torch.inference_mode():
            for images, targets, points, names in tqdm(
                make_loader(valid_dataset, False),
                desc=f"{config['run_name']} epoch {epoch:02d} valid",
            ):
                images = images.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)
                points = points.to(device, non_blocking=True)
                valid = torch.ones(
                    images.shape[0], dtype=torch.bool, device=device
                )

                logits = model(images, points, valid)
                loss = criterion(logits, targets)
                assert torch.isfinite(logits).all()
                assert torch.isfinite(loss)

                count = images.shape[0]
                valid_loss_sum += loss.item() * count
                valid_count += count
                logits_parts.append(logits.cpu().numpy())
                target_parts.append(targets.cpu().numpy())
                validation_names.extend(names)

        assert valid_count == 19867
        assert validation_names == valid_dataset.names

        metrics = evaluate_attributes(
            np.concatenate(logits_parts),
            np.concatenate(target_parts),
            threshold=config["f1_threshold"],
        )
        assert np.isfinite(metrics["macro_f1"])
        assert np.isfinite(metrics["map"])

        torch.cuda.synchronize()
        record = {
            "epoch": epoch,
            "train_bce": train_loss_sum / train_count,
            "validation_bce": valid_loss_sum / valid_count,
            "validation_macro_f1_at_0.5": float(metrics["macro_f1"]),
            "validation_map": float(metrics["map"]),
            "seconds": time.perf_counter() - started,
        }
        history.append(record)
        best = select_best(history)

        checkpoint_path = run_root / f"epoch_{epoch:02d}.pt"
        assert not checkpoint_path.exists(), (
            f"Completed checkpoint already exists: {checkpoint_path}"
        )
        temporary = checkpoint_path.with_name(
            checkpoint_path.name + ".partial"
        )
        torch.save({
            "epoch": epoch,
            "config": config,
            "history": history,
            "best": best,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }, temporary)
        temporary.replace(checkpoint_path)

        write_json_atomic(run_root / "history.json", history)
        write_json_atomic(run_root / "best.json", best)

        print(
            f"Epoch {epoch:02d} saved | "
            f"train BCE={record['train_bce']:.6f} | "
            f"valid BCE={record['validation_bce']:.6f} | "
            f"F1@0.5={metrics['macro_f1']:.6f} | "
            f"mAP={metrics['map']:.6f} | "
            f"{record['seconds'] / 60:.1f} min | "
            f"best epoch={best['epoch']}",
            flush=True,
        )
        del logits_parts, target_parts

    assert len(history) == config["epochs"]
    print(
        "Selected checkpoint:",
        run_root / f"epoch_{best['epoch']:02d}.pt",
    )
    print("Oracle training: COMPLETE")
    print("Validation prediction export and threshold fitting remain pending")
    print("No test evaluation performed")
    return history, best


ORACLE_HISTORY, ORACLE_BEST = train_oracle(
    model=MODEL,
    optimizer=OPTIMIZER,
    criterion=CRITERION,
    config=CONFIG,
    train_dataset=TRAIN_DATASET,
    valid_dataset=VALID_DATASET,
    run_root=RUN_ROOT,
    device=DEVICE,
)

Run: oracle_seed1
Restored: epoch_09.pt
Completed epochs: 9
Next epoch: 10
Geometry: GT; all samples valid


oracle_seed1 epoch 10 train:   0%|          | 0/2544 [00:00<?, ?it/s]

oracle_seed1 epoch 10 valid:   0%|          | 0/311 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7845dc80dc60>
<function _MultiProcessingDataLoaderIter.__del__ at 0x7845dc80dc60>
Exception ignored in: Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__


self._shutdown_workers()      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7845dc80dc60>  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
        if w.is_alive():    
self._shutdown_workers()  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
self._shutdown_workers()    
assert self._

Epoch 10 saved | train BCE=0.041545 | valid BCE=0.366959 | F1@0.5=0.716232 | mAP=0.771491 | 11.3 min | best epoch=3
Selected checkpoint: /content/drive/MyDrive/FacialVisualProfile/runs/oracle_seed1/epoch_03.pt
Oracle training: COMPLETE
Validation prediction export and threshold fitting remain pending
No test evaluation performed


In [5]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
import random
import sys

import numpy as np
import torch
from torch.torch_version import TorchVersion
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from models.resnet18_geometry import ResNet18GeometryAttributes
from evaluation.attributes import evaluate_attributes

SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_NAME = f"oracle_seed{SEED}"
RUN_ROOT = PROJECT / "runs" / RUN_NAME

assert "VALID_DATASET" in globals(), (
    "Run the Oracle preparation or restoration cell first"
)
assert torch.cuda.is_available(), "A GPU runtime is required"


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


config = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
history = json.loads(
    (RUN_ROOT / "history.json").read_text(encoding="utf-8")
)
best = json.loads(
    (RUN_ROOT / "best.json").read_text(encoding="utf-8")
)

assert config["run_name"] == RUN_NAME
assert config["seed"] == SEED
assert [row["epoch"] for row in history] == list(
    range(1, config["epochs"] + 1)
)

selected = max(history, key=lambda row: row["validation_map"])
assert best == {
    "epoch": selected["epoch"],
    "map": selected["validation_map"],
}
checkpoint_path = RUN_ROOT / f"epoch_{best['epoch']:02d}.pt"

environment = {
    "python": platform.python_version(),
    "torch": str(torch.__version__),
    "torchvision": importlib.metadata.version("torchvision"),
    "numpy": importlib.metadata.version("numpy"),
    "Pillow": importlib.metadata.version("Pillow"),
    "scikit-learn": importlib.metadata.version("scikit-learn"),
}
assert environment == config["environment"]
assert {
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": torch.cuda.get_device_name(0),
} == config["runtime"]

for relative, expected in config["source_sha256"].items():
    assert sha_file(RUN_ROOT / "source" / relative) == expected, (
        f"Archived source differs: {relative}"
    )

for module_name, relative in (
    ("models.resnet18_geometry", "models/resnet18_geometry.py"),
    ("evaluation.attributes", "evaluation/attributes.py"),
    ("datasets.celeba", "datasets/celeba.py"),
    ("datasets.preprocessing", "datasets/preprocessing.py"),
):
    assert sha_file(sys.modules[module_name].__file__) == (
        config["source_sha256"][relative]
    ), f"Imported source differs: {module_name}"

assert type(VALID_DATASET).__name__ == "CelebAAttributes"
assert len(VALID_DATASET) == 19867
assert VALID_DATASET.attribute_names == config["attribute_names"]

with torch.serialization.safe_globals([TorchVersion]):
    checkpoint = torch.load(
        checkpoint_path, map_location="cpu", weights_only=True
    )
assert checkpoint["config"] == config
assert checkpoint["epoch"] == best["epoch"]
assert checkpoint["history"] == history[:best["epoch"]]

device = torch.device("cuda")
torch.backends.cudnn.deterministic = config["cudnn_deterministic"]
torch.backends.cudnn.benchmark = config["cudnn_benchmark"]

export_model = ResNet18GeometryAttributes(pretrained=False).to(device)
export_model.load_state_dict(
    checkpoint["model_state_dict"], strict=True
)
del checkpoint
export_model.eval()

loader = DataLoader(
    VALID_DATASET,
    batch_size=config["batch_size"],
    shuffle=False,
    num_workers=config["num_workers"],
    worker_init_fn=seed_worker,
    pin_memory=True,
    drop_last=False,
)

logits_parts = []
target_parts = []
filenames = []

with torch.inference_mode():
    for images, targets, points, names in tqdm(
        loader, desc=f"{RUN_NAME} selected validation"
    ):
        images = images.to(device, non_blocking=True)
        points = points.to(device, non_blocking=True)
        valid = torch.ones(
            len(names), dtype=torch.bool, device=device
        )
        logits = export_model(images, points, valid)
        assert logits.shape == (len(names), 24)
        assert torch.isfinite(logits).all()

        logits_parts.append(logits.cpu().numpy())
        target_parts.append(targets.numpy())
        filenames.extend(names)

payload = {
    "filenames": np.asarray(filenames),
    "logits": np.concatenate(logits_parts),
    "targets": np.concatenate(target_parts),
    "geometry_valid": np.ones(len(filenames), dtype=np.bool_),
    "attribute_names": np.asarray(config["attribute_names"]),
}

assert filenames == VALID_DATASET.names
assert payload["logits"].shape == payload["targets"].shape == (19867, 24)
assert payload["geometry_valid"].all()

reference_path = (
    PROJECT / "runs/r18_seed0/day7/validation_predictions.npz"
)
with np.load(reference_path, allow_pickle=False) as reference:
    for key in ("filenames", "targets", "attribute_names"):
        assert np.array_equal(payload[key], reference[key]), (
            f"Historical validation alignment differs: {key}"
        )

metrics = evaluate_attributes(
    payload["logits"], payload["targets"], threshold=0.5
)
assert abs(
    metrics["macro_f1"] - selected["validation_macro_f1_at_0.5"]
) <= 1e-10, "Selected-checkpoint F1 reproduction failed"
assert abs(
    metrics["map"] - best["map"]
) <= 1e-10, "Selected-checkpoint mAP reproduction failed"

prediction_path = RUN_ROOT / "validation_predictions.npz"
if prediction_path.exists():
    with np.load(prediction_path, allow_pickle=False) as existing:
        assert set(existing.files) == set(payload)
        for key, value in payload.items():
            assert np.array_equal(existing[key], value), (
                f"Existing predictions differ; preserved: {key}"
            )
else:
    temporary = prediction_path.with_name(
        prediction_path.name + ".partial"
    )
    with temporary.open("wb") as stream:
        np.savez_compressed(stream, **payload)
    temporary.replace(prediction_path)

audit = {
    "run_name": RUN_NAME,
    "checkpoint": checkpoint_path.name,
    "selected_epoch": best["epoch"],
    "checkpoint_sha256": sha_file(checkpoint_path),
    "config_sha256": sha_file(RUN_ROOT / "config.json"),
    "prediction_sha256": sha_file(prediction_path),
    "split": "valid",
    "samples": 19867,
    "masked_validation_samples": 0,
    "attribute_names": config["attribute_names"],
    "macro_f1_at_0.5": float(metrics["macro_f1"]),
    "map": float(metrics["map"]),
    "source_sha256": config["source_sha256"],
    "passed": True,
    "threshold_fitting_performed": False,
    "test_evaluated": False,
}
audit_path = RUN_ROOT / "validation_export_audit.json"
if audit_path.exists():
    assert json.loads(audit_path.read_text(encoding="utf-8")) == audit, (
        "Existing export audit differs; preserved"
    )
else:
    temporary = audit_path.with_name(audit_path.name + ".partial")
    temporary.write_text(
        json.dumps(audit, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(audit_path)

print("Full validation IDs, targets and attribute order: PASS")
print("GT geometry valid for all validation samples: PASS")
print("Selected-checkpoint metric reproduction: PASS")
print(f"F1@0.5: {metrics['macro_f1']:.6f}")
print(f"mAP: {metrics['map']:.6f}")
print("Saved:", prediction_path)
print("Oracle selected-validation prediction export: PASS")
print("No training, threshold fitting or test evaluation performed")

del export_model, loader, logits_parts, target_parts
torch.cuda.empty_cache()

oracle_seed1 selected validation:   0%|          | 0/311 [00:00<?, ?it/s]

Full validation IDs, targets and attribute order: PASS
GT geometry valid for all validation samples: PASS
Selected-checkpoint metric reproduction: PASS
F1@0.5: 0.735095
mAP: 0.819262
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/oracle_seed1/validation_predictions.npz
Oracle selected-validation prediction export: PASS
No training, threshold fitting or test evaluation performed


In [6]:
from pathlib import Path
import hashlib
import json

import numpy as np
from sklearn.metrics import average_precision_score, f1_score

SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
RUN_NAME = f"oracle_seed{SEED}"
RUN_ROOT = PROJECT / "runs" / RUN_NAME
PREDICTION_PATH = RUN_ROOT / "validation_predictions.npz"
OUTPUT_PATH = (
    PROJECT / "runs/benchmark_preparation"
    / f"seed{SEED}" / f"{RUN_NAME}.json"
)


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


config = json.loads(
    (RUN_ROOT / "config.json").read_text(encoding="utf-8")
)
audit = json.loads(
    (RUN_ROOT / "validation_export_audit.json").read_text(
        encoding="utf-8"
    )
)
best = json.loads(
    (RUN_ROOT / "best.json").read_text(encoding="utf-8")
)

assert config["run_name"] == RUN_NAME
assert config["seed"] == SEED
assert audit["run_name"] == RUN_NAME
assert audit["passed"] is True
assert audit["split"] == "valid"
assert audit["test_evaluated"] is False
assert audit["selected_epoch"] == best["epoch"]
assert audit["checkpoint"] == f"epoch_{best['epoch']:02d}.pt"
assert audit["masked_validation_samples"] == 0

assert sha_file(RUN_ROOT / "config.json") == audit["config_sha256"]
assert sha_file(PREDICTION_PATH) == audit["prediction_sha256"]

for relative, expected in config["source_sha256"].items():
    assert sha_file(RUN_ROOT / "source" / relative) == expected, (
        f"Archived source differs: {relative}"
    )

with np.load(PREDICTION_PATH, allow_pickle=False) as saved:
    filenames = saved["filenames"].copy()
    logits = saved["logits"].copy()
    targets = saved["targets"].copy()
    geometry_valid = saved["geometry_valid"].copy()
    attribute_names = saved["attribute_names"].tolist()

assert logits.shape == targets.shape == (19867, 24)
assert filenames.shape == (19867,)
assert len(set(filenames.tolist())) == 19867
assert np.isfinite(logits).all()
assert np.isin(targets, [0, 1]).all()
assert attribute_names == config["attribute_names"]
assert geometry_valid.shape == (19867,)
assert geometry_valid.dtype == np.bool_
assert geometry_valid.all()
assert np.all((targets.sum(axis=0) > 0) & (targets.sum(axis=0) < 19867))

reference_path = (
    PROJECT / "runs/r18_seed0/day7/validation_predictions.npz"
)
with np.load(reference_path, allow_pickle=False) as reference:
    assert np.array_equal(filenames, reference["filenames"])
    assert np.array_equal(targets, reference["targets"])
    assert attribute_names == reference["attribute_names"].tolist()

print("Historical validation IDs, targets and attribute order: PASS")

# Match the historical stable float64 sigmoid.
values = logits.astype(np.float64)
probabilities = np.empty_like(values)
positive = values >= 0
probabilities[positive] = 1.0 / (1.0 + np.exp(-values[positive]))
exp_values = np.exp(values[~positive])
probabilities[~positive] = exp_values / (1.0 + exp_values)

grid_integer = list(range(5, 96))
grid = [value / 100 for value in grid_integer]
thresholds = []
baseline_f1 = []
tuned_f1 = []
attribute_ap = []

for column in range(24):
    labels = targets[:, column]
    scores = probabilities[:, column]

    baseline_f1.append(float(f1_score(
        labels, scores >= 0.5, zero_division=0
    )))
    attribute_ap.append(float(average_precision_score(labels, scores)))

    candidate_f1 = [
        float(f1_score(labels, scores >= threshold, zero_division=0))
        for threshold in grid
    ]

    # Exact ties prefer the threshold closest to 0.5, then the lower one.
    selected = min(
        range(len(grid)),
        key=lambda index: (
            -candidate_f1[index],
            abs(grid_integer[index] - 50),
            grid_integer[index],
        ),
    )
    thresholds.append(grid[selected])
    tuned_f1.append(candidate_f1[selected])

baseline_macro_f1 = float(np.mean(baseline_f1))
tuned_macro_f1 = float(np.mean(tuned_f1))
mean_ap = float(np.mean(attribute_ap))

assert abs(baseline_macro_f1 - audit["macro_f1_at_0.5"]) <= 1e-10
assert abs(mean_ap - audit["map"]) <= 1e-10
assert abs(mean_ap - best["map"]) <= 1e-10
assert all(
    tuned >= baseline
    for tuned, baseline in zip(tuned_f1, baseline_f1)
)

print("Selected-checkpoint metric reproduction: PASS")
print("Computing checkpoint SHA256...", flush=True)
checkpoint_hash = sha_file(RUN_ROOT / audit["checkpoint"])
assert checkpoint_hash == audit["checkpoint_sha256"]

report = {
    "run_name": RUN_NAME,
    "checkpoint": audit["checkpoint"],
    "selected_epoch": audit["selected_epoch"],
    "checkpoint_sha256": checkpoint_hash,
    "config_sha256": audit["config_sha256"],
    "prediction_path": str(PREDICTION_PATH),
    "prediction_sha256": audit["prediction_sha256"],
    "fit_split": "valid",
    "samples": 19867,
    "masked_validation_samples": 0,
    "geometry_pipeline": config["geometry_source"],
    "attribute_names": attribute_names,
    "thresholds": thresholds,
    "objective": "per_attribute_F1",
    "grid": grid,
    "tie_break": "closest_to_0.5_then_lower",
    "comparison": ">=",
    "baseline_macro_f1": baseline_macro_f1,
    "tuned_macro_f1": tuned_macro_f1,
    "map": mean_ap,
    "per_attribute_f1_at_0.5": baseline_f1,
    "per_attribute_f1_tuned": tuned_f1,
    "per_attribute_ap": attribute_ap,
    "source_sha256": config["source_sha256"],
    "environment": config["environment"],
    "test_evaluated": False,
    "note": (
        "Thresholds fitted on clean validation only. GT geometry is "
        "valid for every sample. Tuned validation F1 is an in-sample "
        "fitting result, not independent evaluation. Reuse these "
        "thresholds unchanged for all clean and corrupted test conditions."
    ),
}

if OUTPUT_PATH.exists():
    assert json.loads(
        OUTPUT_PATH.read_text(encoding="utf-8")
    ) == report, f"Existing threshold report differs; preserved: {OUTPUT_PATH}"
else:
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary = OUTPUT_PATH.with_name(OUTPUT_PATH.name + ".partial")
    temporary.write_text(
        json.dumps(report, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(OUTPUT_PATH)

print("Run:", RUN_NAME)
print(f"F1@0.5: {baseline_macro_f1:.6f}")
print(f"F1(clean-validation tuned): {tuned_macro_f1:.6f}")
print(f"mAP: {mean_ap:.6f}")
print("Frozen thresholds:", len(thresholds))
print("Saved:", OUTPUT_PATH)
print("Oracle clean-validation threshold freeze: PASS")
print("No model inference, training or test evaluation performed")

Historical validation IDs, targets and attribute order: PASS
Selected-checkpoint metric reproduction: PASS
Computing checkpoint SHA256...
Run: oracle_seed1
F1@0.5: 0.735095
F1(clean-validation tuned): 0.763577
mAP: 0.819262
Frozen thresholds: 24
Saved: /content/drive/MyDrive/FacialVisualProfile/runs/benchmark_preparation/seed1/oracle_seed1.json
Oracle clean-validation threshold freeze: PASS
No model inference, training or test evaluation performed


In [7]:
from google.colab import files
from pathlib import Path
import hashlib
import json
import shutil
import zipfile

SEED = 1
PROJECT = Path("/content/drive/MyDrive/FacialVisualProfile")
ARCHIVE_PATH = Path(f"/content/seed{SEED}_training_results_archive.zip")
DRIVE_PATH = (
    PROJECT / "runs/training_archives" / ARCHIVE_PATH.name
)


def sha_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


payloads = {}
summary = []

for model_name in ("r18", "r50", "m3", "oracle"):
    run_name = f"{model_name}_seed{SEED}"
    run_root = PROJECT / "runs" / run_name
    threshold_path = (
        PROJECT / "runs/benchmark_preparation"
        / f"seed{SEED}" / f"{run_name}.json"
    )

    config = json.loads(
        (run_root / "config.json").read_text(encoding="utf-8")
    )
    history = json.loads(
        (run_root / "history.json").read_text(encoding="utf-8")
    )
    threshold = json.loads(
        threshold_path.read_text(encoding="utf-8")
    )
    audit = json.loads(
        (run_root / "validation_export_audit.json").read_text(
            encoding="utf-8"
        )
    )

    assert config["seed"] == SEED
    assert config["run_name"] == run_name
    assert [row["epoch"] for row in history] == list(
        range(1, config["epochs"] + 1)
    )
    assert all(
        (run_root / f"epoch_{epoch:02d}.pt").is_file()
        for epoch in range(config["epochs"] + 1)
    )
    assert audit["passed"] is True
    assert audit["test_evaluated"] is False
    assert threshold["run_name"] == run_name
    assert threshold["fit_split"] == "valid"
    assert threshold["samples"] == 19867
    assert threshold["test_evaluated"] is False
    assert threshold["attribute_names"] == config["attribute_names"]
    assert len(threshold["thresholds"]) == 24
    assert threshold["grid"] == [i / 100 for i in range(5, 96)]
    assert threshold["tie_break"] == "closest_to_0.5_then_lower"
    assert threshold["comparison"] == ">="
    assert all(
        value in threshold["grid"] for value in threshold["thresholds"]
    )

    print(f"{run_name}: verifying file bindings...", flush=True)
    for path, key in (
        (run_root / "config.json", "config_sha256"),
        (run_root / threshold["checkpoint"], "checkpoint_sha256"),
        (run_root / "validation_predictions.npz", "prediction_sha256"),
    ):
        actual = sha_file(path)
        assert actual == threshold[key] == audit[key], (
            f"File binding differs: {path}"
        )
    assert threshold["selected_epoch"] == audit["selected_epoch"]
    assert threshold["checkpoint"] == audit["checkpoint"]

    for relative, expected in config["source_sha256"].items():
        source = run_root / "source" / relative
        assert sha_file(source) == expected, (
            f"Archived source differs: {source}"
        )
        payloads[f"runs/{run_name}/source/{relative}"] = (
            source.read_bytes()
        )

    for filename in (
        "config.json", "history.json", "best.json",
        "validation_export_audit.json", "data_preparation.json",
    ):
        path = run_root / filename
        if path.is_file():
            payloads[f"runs/{run_name}/{filename}"] = path.read_bytes()

    threshold_relative = threshold_path.relative_to(PROJECT).as_posix()
    payloads[threshold_relative] = threshold_path.read_bytes()

    summary.append({
        "run_name": run_name,
        "selected_epoch": threshold["selected_epoch"],
        "checkpoint": threshold["checkpoint"],
        "f1_at_0.5": threshold["baseline_macro_f1"],
        "clean_validation_tuned_f1": threshold["tuned_macro_f1"],
        "map": threshold["map"],
        "checkpoint_sha256": threshold["checkpoint_sha256"],
        "prediction_sha256": threshold["prediction_sha256"],
    })
    print(f"{run_name}: training records and frozen thresholds PASS")

payloads["summary.json"] = (
    json.dumps(summary, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

manifest = {
    "seed": SEED,
    "scope": "Completed training and clean-validation threshold preparation",
    "files": {
        name: hashlib.sha256(data).hexdigest()
        for name, data in sorted(payloads.items())
    },
    "checkpoints_included": False,
    "prediction_npz_included": False,
    "images_included": False,
    "test_evaluated": False,
}
payloads["archive_manifest.json"] = (
    json.dumps(manifest, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

# Fixed ZIP metadata makes identical reruns produce identical archives.
temporary = ARCHIVE_PATH.with_name(ARCHIVE_PATH.name + ".partial")
with zipfile.ZipFile(temporary, "w") as archive:
    for name, data in sorted(payloads.items()):
        info = zipfile.ZipInfo(name, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        archive.writestr(info, data)

with zipfile.ZipFile(temporary) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(payloads)
    for name, data in payloads.items():
        assert archive.read(name) == data

temporary.replace(ARCHIVE_PATH)
archive_hash = sha_file(ARCHIVE_PATH)

DRIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_PATH.exists():
    assert sha_file(DRIVE_PATH) == archive_hash, (
        f"Existing archive differs; preserved: {DRIVE_PATH}"
    )
else:
    drive_temporary = DRIVE_PATH.with_name(DRIVE_PATH.name + ".partial")
    shutil.copyfile(ARCHIVE_PATH, drive_temporary)
    assert sha_file(drive_temporary) == archive_hash
    drive_temporary.replace(DRIVE_PATH)

print("Four-model training archive: PASS")
print("Archive SHA256:", archive_hash)
print("Saved on Drive:", DRIVE_PATH)
print("No training, threshold refitting or test evaluation performed")
files.download(str(ARCHIVE_PATH))

r18_seed1: verifying file bindings...
r18_seed1: training records and frozen thresholds PASS
r50_seed1: verifying file bindings...
r50_seed1: training records and frozen thresholds PASS
m3_seed1: verifying file bindings...
m3_seed1: training records and frozen thresholds PASS
oracle_seed1: verifying file bindings...
oracle_seed1: training records and frozen thresholds PASS
Four-model training archive: PASS
Archive SHA256: de1ac9a050ded352a903b601050b65f10be469a14a50c33824cbe3b02dfd8b9c
Saved on Drive: /content/drive/MyDrive/FacialVisualProfile/runs/training_archives/seed1_training_results_archive.zip
No training, threshold refitting or test evaluation performed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>